# Aprendizaje y Clasificación Automática con R

**Autor:** Jesús Gilberto Rodríguez Escobedo

Este es el **cuaderno maestro completo**. Para trabajar de forma más ligera se recomienda usar los cuadernos autónomos por capítulo:

[Abrir índice de Colabs por capítulo](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)

## Cómo ejecutar este cuaderno completo

1. Ejecute primero la celda de preparación automática.
2. Ejecute las celdas en orden, de arriba hacia abajo.
3. Si Colab reinicia o desconecta el entorno, vuelva a ejecutar desde la primera celda.


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Índice general

- Capítulo 1. Introducción al aprendizaje automático
- Capítulo 2. Preparación de datos reales
- Capítulo 3. Análisis exploratorio de datos
- Capítulo 4. Regresión lineal simple y múltiple
- Capítulo 5. Regresión logística
- Capítulo 6. k vecinos más cercanos (k-NN)
- Capítulo 7. Árboles de decisión
- Capítulo 8. Random Forest
- Capítulo 9. Evaluación y comparación de modelos
- Capítulo 10. Máquinas de vectores de soporte (SVM)
- Capítulo 11. Naive Bayes
- Capítulo 12. Redes neuronales
- Capítulo 13. Agrupamiento k-means


# ¿Qué es el aprendizaje automático?

La formulación matemática de **Formulación del aprendizaje supervisado y teoría de clasificación** se desarrolla con mayor profundidad
en los capítulos 5 y 6 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar qué es el aprendizaje automático, diferenciarlo de la programación tradicional e identificar problemas de clasificación, regresión y agrupamiento.

## Introducción

El aprendizaje automático, también conocido como *Machine Learning*, permite construir modelos capaces de aprender patrones a partir de datos.

En la programación tradicional, una persona define reglas. En aprendizaje automático, proporcionamos ejemplos para que un algoritmo aprenda una relación entre variables de entrada y una salida esperada.

## Programación tradicional contra aprendizaje automático


In [ ]:
pm10 <- 85

if (pm10 > 75) {
  print("Contaminación alta")
} else {
  print("Contaminación baja")
}


## Explicación del código
Se define un valor de PM10 y después se aplica una regla fija. Si el valor es mayor que 75, el programa clasifica el día como de contaminación alta.

## Interpretación del resultado
Este ejemplo representa programación tradicional: la decisión depende de una regla escrita manualmente. En aprendizaje automático, la regla se aprende a partir de ejemplos.


In [ ]:
datos <- data.frame(
  dia = 1:12,
  pm10 = c(42, 88, 55, 120, 63, 95, 38, 110, 72, 130, 47, 99),
  temperatura = c(25, 28, 22, 30, 24, 29, 21, 31, 27, 32, 23, 30),
  humedad = c(40, 35, 60, 30, 55, 33, 65, 29, 42, 28, 62, 34),
  clase = c("Baja", "Alta", "Baja", "Alta", "Baja", "Alta", "Baja", "Alta", "Baja", "Alta", "Baja", "Alta")
)

datos


## Explicación del código
Se construye una pequeña base de datos con 12 días. Cada fila contiene variables ambientales y una clase conocida.

## Variables predictoras y variable respuesta

La variable respuesta es la variable que queremos predecir. Las variables predictoras son las variables que usamos como entrada para el modelo.

## Tipos principales de aprendizaje automático

1. Aprendizaje supervisado.
2. Aprendizaje no supervisado.
3. Aprendizaje por refuerzo.

## Notación básica

Supongamos que tenemos $n$ observaciones y $p$ variables predictoras. Podemos representar los datos mediante una matriz $X$ y una variable respuesta $Y$.

$$
\hat{y} = f(x_1, x_2, \ldots, x_p)
$$

## Primer ejemplo visual en R


In [ ]:
library(ggplot2)
source("util_graficas.R")

ggplot(datos, aes(x = pm10, y = temperatura, color = clase)) +
  geom_point(size = 4, alpha = 0.9) +
  escala_clases_color() +
  labs(
    title = "Ejemplo inicial de clasificación",
    subtitle = "Clasificación de contaminación alta o baja",
    x = "PM10",
    y = "Temperatura",
    color = "Clase"
  ) +
  tema_libro()


## Explicación del código
La gráfica coloca PM10 en el eje horizontal, temperatura en el eje vertical y usa color para distinguir la clase.

## Interpretación del resultado
Los puntos permiten visualizar si las clases se separan. Esta es la idea básica detrás de muchos métodos de clasificación.

## División entre entrenamiento y prueba


In [ ]:
set.seed(123)
n <- nrow(datos)
indices_entrenamiento <- sample(1:n, size = round(0.7 * n))

entrenamiento <- datos[indices_entrenamiento, ]
prueba <- datos[-indices_entrenamiento, ]

entrenamiento
prueba


## Explicación del código
Se divide la base en dos partes: una para entrenar el modelo y otra para evaluar cómo funciona con datos no usados durante el ajuste.

## Primer clasificador basado en una regla


In [ ]:
datos$prediccion_regla <- ifelse(datos$pm10 > 75, "Alta", "Baja")

tabla_confusion <- table(
  Real = datos$clase,
  Predicho = datos$prediccion_regla
)

tabla_confusion
mean(datos$clase == datos$prediccion_regla)


## Interpretación del resultado
La matriz de confusión compara la clase real contra la clase predicha. La exactitud mide la proporción de aciertos.

## Materiales complementarios del capítulo
Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual y el video explica los contenidos de manera audiovisual.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Introducción audiovisual al aprendizaje automático y a los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=N2SkzjT2LpU) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico.pdf) | [Descargar PDF](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico.pdf){download="capitulo-01-introduccion-aprendizaje-automatico.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico.pptx) | [Descargar PPTX](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico.pptx){download="capitulo-01-introduccion-aprendizaje-automatico.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico-infografia.png) | [Descargar PNG](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico-infografia.png){download="capitulo-01-introduccion-aprendizaje-automatico-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/01-introduccion.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 1](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico-infografia.png)](recursos/capitulo-01/capitulo-01-introduccion-aprendizaje-automatico-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/01-introduccion.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=N2SkzjT2LpU>

La presentación PDF, el archivo editable y la infografía pueden descargarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Laboratorio interactivo: una regla de clasificación

Este laboratorio muestra cómo una regla sencilla puede clasificar observaciones
a partir de un umbral.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

Permite cambiar el umbral de una regla de clasificación y observar la exactitud.

## Conclusión

El aprendizaje automático permite construir modelos que aprenden patrones a partir de datos. En este capítulo vimos la diferencia entre reglas manuales y aprendizaje a partir de ejemplos.


# Preparación de datos reales

La formulación matemática de **representación matemática de datos, probabilidad y estadística** se desarrolla con mayor profundidad
en los capítulos 1, 2 y 3 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de:

- localizar y descargar la base ATUS desde el portal oficial del INEGI;
- organizar los archivos dentro del proyecto Quarto;
- leer y revisar una base real en R;
- seleccionar y transformar variables;
- construir una variable respuesta binaria;
- guardar una base preparada para los capítulos de modelado;
- citar correctamente la fuente de los datos.

## Introducción

En la práctica profesional, los datos casi nunca llegan preparados para aplicar un algoritmo. Antes de modelar debemos conocer su procedencia, revisar su estructura, limpiar valores, transformar variables y documentar cada decisión.

En este libro utilizamos la **Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS)** del Instituto Nacional de Estadística y Geografía [@inegi_atus_2024]. Su objetivo es producir información anual sobre la siniestralidad del transporte terrestre en zonas de jurisdicción no federal, con desglose nacional, estatal y municipal.

Los datos originales son del INEGI. La limpieza, transformación, selección de variables, gráficas, modelos e interpretaciones de este libro son elaboración del autor. No constituyen resultados oficiales ni implican el aval del Instituto.

## Descargar la base ATUS desde el INEGI

La descarga debe realizarse desde el sitio oficial para conservar la procedencia y los metadatos del archivo.

1. Abra en su navegador el portal de **Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS)**:

   <https://www.inegi.org.mx/programas/accidentes/>

2. Localice el apartado **Datos abiertos** o la opción de descarga correspondiente.
3. Seleccione el periodo anual que desea utilizar. Para reproducir los ejemplos de esta versión, elija **2024**.
4. Descargue el archivo en formato CSV. Es posible que el portal entregue un archivo comprimido en formato ZIP.
5. Descomprima el archivo descargado.
6. Identifique el CSV anual y cópielo en la carpeta `datos` del proyecto.
7. Para seguir exactamente los ejemplos, renómbrelo como:

```text
atus_2024.csv
```

La estructura mínima del proyecto debe quedar así:

```text
libro-machine-learning-r/
├── _quarto.yml
├── index.qmd
├── 02-preparacion-datos.qmd
├── datos/
│   └── atus_2024.csv
└── referencias.bib
```

Puede utilizar un año diferente. En ese caso, cambie el nombre del archivo o modifique la ruta utilizada en el código. Conviene anotar siempre el año de referencia porque los resultados pueden cambiar entre periodos.

## Cómo citar los datos

Los términos de libre uso del INEGI permiten utilizar, adaptar y publicar su información, pero requieren citar la fuente de origen [@inegi_terminos_libre_uso].

En el texto puede utilizarse una cita como esta:

> Los datos proceden de la Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas del INEGI [@inegi_atus_2024].

Debajo de una tabla o gráfica sin transformaciones importantes:

> **Fuente:** INEGI, Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), 2024.

Cuando exista procesamiento, agrupación, modelado o visualización propia:

> **Fuente:** Elaboración propia con datos del INEGI, Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), 2024.

## Cargar paquetes


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
library(stringr)
source("util_graficas.R")


Se cargan paquetes para leer archivos CSV, transformar datos, trabajar con texto y crear gráficas. El archivo `util_graficas.R` contiene funciones visuales comunes para mantener un estilo uniforme en todo el libro.

## Localizar el archivo

En lugar de depender únicamente de un nombre fijo, el siguiente código busca archivos cuyo nombre comience con `atus` y termine en `.csv`.


In [ ]:
archivos_atus <- list.files(
  path = "datos",
  pattern = "^atus.*\\.csv$",
  full.names = TRUE,
  ignore.case = TRUE
)

# Preferir una base ATUS cruda cuando esté disponible.
# La base *_ml_preparado.csv es una salida de este mismo capítulo y
# no contiene necesariamente CLASACC.
archivos_crudos <- archivos_atus[
  !grepl("ml_preparado", basename(archivos_atus), ignore.case = TRUE)
]

archivos_atus


Si aparece al menos una ruta, R encontró un archivo ATUS. Si el resultado es `character(0)`, revise que el archivo esté descomprimido y guardado dentro de la carpeta `datos`.

## Leer el archivo CSV


In [ ]:
if (length(archivos_atus) == 0) {
  stop(
    paste(
      "No se encontró un archivo ATUS en la carpeta datos.",
      "Descárguelo desde el portal oficial del INEGI y descomprímalo."
    )
  )
}

ruta_atus <- if (length(archivos_crudos) > 0) {
  archivos_crudos[1]
} else {
  archivos_atus[1]
}

atus <- read_csv(
  file = ruta_atus,
  show_col_types = FALSE,
  locale = locale(encoding = "UTF-8")
)

message("Archivo leído: ", ruta_atus)


Se selecciona el primer archivo localizado y se lee con `read_csv()`. El argumento `show_col_types = FALSE` evita imprimir mensajes técnicos que distraen de los resultados principales.

## Primer vistazo a los datos


In [ ]:
resumen_dimensiones <- data.frame(
  filas = nrow(atus),
  columnas = ncol(atus)
)

resumen_dimensiones
head(atus)


Las filas representan registros de accidentes y las columnas describen sus características temporales, geográficas y operativas. Antes de modelar es indispensable confirmar que las variables esperadas estén disponibles.

## Normalizar los nombres de variables


In [ ]:
names(atus) <- toupper(names(atus))
names(atus)


Se convierten los nombres a mayúsculas para evitar errores por diferencias como `Mes`, `MES` o `mes`.

## Seleccionar variables de interés


In [ ]:
variables_interes <- c(
  "ANIO", "MES", "ID_ENTIDAD", "ID_MUNICIPIO",
  "ID_HORA", "ID_DIA", "DIASEMANA", "TIPACCID",
  "CAUSAACCI", "CLASACC"
)

variables_existentes <- intersect(
  variables_interes,
  names(atus)
)

variables_faltantes <- setdiff(
  variables_interes,
  names(atus)
)

atus_base <- atus |>
  select(all_of(variables_existentes))

variables_existentes
variables_faltantes


Los nombres y categorías pueden variar entre versiones. Si aparece una variable faltante, consulte los metadatos y el diccionario de datos descargados junto con la base antes de sustituirla.

## Crear la variable respuesta

El primer problema de clasificación distinguirá entre:

- **Con víctimas:** accidentes fatales o no fatales con personas lesionadas o fallecidas.
- **Solo daños:** accidentes con daños materiales sin víctimas registradas.


In [ ]:
if ("CLASACC" %in% names(atus_base)) {

  # Ruta A: base ATUS cruda del INEGI.
  atus_modelo <- atus_base |>
    mutate(
      CLASACC_TXT = str_to_lower(as.character(CLASACC)),
      accidente_con_victimas = case_when(
        str_detect(CLASACC_TXT, "sólo daños") ~ "Solo daños",
        str_detect(CLASACC_TXT, "solo daños") ~ "Solo daños",
        str_detect(CLASACC_TXT, "daños") ~ "Solo daños",
        str_detect(CLASACC_TXT, "no fatal") ~ "Con víctimas",
        str_detect(CLASACC_TXT, "fatal") ~ "Con víctimas",
        TRUE ~ NA_character_
      )
    ) |>
    filter(!is.na(accidente_con_victimas))

} else if ("ACCIDENTE_CON_VICTIMAS" %in% names(atus)) {

  # Ruta B: base ya preparada incluida con el libro.
  # Después de normalizar nombres, accidente_con_victimas aparece en mayúsculas.
  atus_modelo <- atus |>
    transmute(
      MES,
      ID_HORA,
      DIASEMANA,
      TIPACCID,
      CAUSAACCI,
      accidente_con_victimas = as.character(ACCIDENTE_CON_VICTIMAS)
    ) |>
    mutate(
      accidente_con_victimas = case_when(
        str_to_lower(accidente_con_victimas) %in% c(
          "con víctimas", "con victimas"
        ) ~ "Con víctimas",
        str_to_lower(accidente_con_victimas) %in% c(
          "solo daños", "sólo daños"
        ) ~ "Solo daños",
        TRUE ~ NA_character_
      )
    ) |>
    filter(!is.na(accidente_con_victimas))

  message(
    "Se utilizó datos/atus_ml_preparado.csv como respaldo para el render. ",
    "Para reproducir desde cero la preparación, use la base ATUS cruda del INEGI."
  )

} else {
  stop(
    paste(
      "No se encontró CLASACC ni ACCIDENTE_CON_VICTIMAS.",
      "Revise el diccionario y la versión de la base ATUS."
    )
  )
}

table(atus_modelo$accidente_con_victimas)


La variable `CLASACC` se transforma en una respuesta binaria. `case_when()` asigna una nueva categoría según el texto encontrado y los registros que no pueden clasificarse se excluyen temporalmente.

## Preparar las variables finales


In [ ]:
atus_ml <- atus_modelo |>
  mutate(
    MES = as.factor(MES),
    ID_HORA = as.character(ID_HORA),
    DIASEMANA = as.factor(DIASEMANA),
    TIPACCID = as.factor(TIPACCID),
    CAUSAACCI = as.factor(CAUSAACCI),
    accidente_con_victimas = factor(
      accidente_con_victimas,
      levels = c("Con víctimas", "Solo daños")
    )
  ) |>
  select(
    accidente_con_victimas,
    MES,
    ID_HORA,
    DIASEMANA,
    TIPACCID,
    CAUSAACCI
  ) |>
  na.omit()

data.frame(
  filas = nrow(atus_ml),
  columnas = ncol(atus_ml)
)


La base final queda lista para el análisis exploratorio y el modelado. Cada fila representa un accidente y cada columna una variable predictora o la respuesta que se desea clasificar.

## Guardar la base preparada


In [ ]:
if (!dir.exists("datos")) {
  dir.create("datos")
}

write_csv(
  atus_ml,
  "datos/atus_ml_preparado.csv"
)


Guardar la base preparada evita repetir toda la limpieza en los capítulos siguientes y ayuda a mantener reproducible el flujo de trabajo.

## Lista de comprobación

Antes de continuar, verifique que:

- el archivo proviene del portal oficial del INEGI;
- el año de los datos está documentado;
- el CSV se encuentra dentro de `datos`;
- las variables usadas existen en la versión descargada;
- la transformación de `CLASACC` corresponde con su diccionario;
- las tablas y gráficas incluyen la fuente;
- el archivo `atus_ml_preparado.csv` se creó correctamente.

## Resumen del capítulo

En este capítulo descargamos y documentamos una base real del INEGI, verificamos su ubicación, seleccionamos variables, construimos una respuesta binaria y guardamos una base preparada para aprendizaje automático.

## Materiales complementarios del capítulo
Estos recursos permiten repasar la preparación de datos reales mediante una presentación, una infografía y un video explicativo.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual del proceso de descarga, revisión, limpieza y preparación de datos. | [Ver en YouTube](https://www.youtube.com/watch?v=bmv-tDPZGjQ) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pdf) | [Descargar PDF](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pdf){download="capitulo-02-preparacion-datos-reales.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pptx) | [Descargar PPTX](recursos/capitulo-02/capitulo-02-preparacion-datos-reales.pptx){download="capitulo-02-preparacion-datos-reales.pptx"} |
| Infografía | Resumen visual de las etapas de preparación de datos. | [Ver infografía](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png) | [Descargar PNG](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png){download="capitulo-02-preparacion-datos-reales-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/02-preparacion-datos.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 2](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png)](recursos/capitulo-02/capitulo-02-preparacion-datos-reales-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/02-preparacion-datos.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=bmv-tDPZGjQ>

La presentación PDF, el archivo editable y la infografía pueden descargarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Laboratorio interactivo: preparar datos

Explora valores faltantes, imputación y estandarización.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

Permite introducir valores faltantes, imputarlos y estandarizar las variables.

## Caso aplicado B: preparación de datos COVID-19

En esta segunda ruta aplicada utilizamos una muestra nacional de **50 000
registros confirmados de COVID-19 en México durante 2022**. El año fue
seleccionado después de comparar los cierres históricos que conservan un
esquema homologable.

La base preparada se encuentra en:

```text
datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz
```

### Lectura de la base


In [ ]:
library(readr)
library(dplyr)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

dim(covid)
names(covid)


### Variables binarias y códigos especiales

En la base preparada, las variables clínicas de tipo sí/no fueron
recodificadas así:

| Valor | Interpretación |
|---:|---|
| 0 | No |
| 1 | Sí |
| `NA` | No especificado, se ignora o no aplica |

La variable `TIPO_PACIENTE` quedó codificada como:

| Valor | Interpretación |
|---:|---|
| 0 | Ambulatorio |
| 1 | Hospitalizado |

La variable objetivo `MURIO` se derivó de `FECHA_DEF`:

| Valor | Interpretación |
|---:|---|
| 0 | Sin defunción registrada |
| 1 | Defunción registrada |

### Revisión de valores faltantes


In [ ]:
faltantes_covid <- covid |>
  summarise(
    across(
      everything(),
      ~ sum(is.na(.x))
    )
  ) |>
  tidyr::pivot_longer(
    cols = everything(),
    names_to = "variable",
    values_to = "faltantes"
  ) |>
  arrange(desc(faltantes))

head(faltantes_covid, 10)


### Número de comorbilidades

La variable `NUM_COMORBILIDADES` resume la presencia de:

- diabetes;
- EPOC;
- asma;
- inmunosupresión;
- hipertensión;
- enfermedad cardiovascular;
- obesidad;
- enfermedad renal crónica;
- tabaquismo.


In [ ]:
covid |>
  count(NUM_COMORBILIDADES) |>
  arrange(NUM_COMORBILIDADES)


Estos datos se emplean con fines educativos. Los registros administrativos
pueden contener sesgos, valores desconocidos y diferencias de cobertura. Los
modelos construidos con esta base no sustituyen una valoración médica.


# Análisis exploratorio de datos

La formulación matemática de **probabilidad, estadística, covarianza y representación multivariada** se desarrolla con mayor profundidad
en los capítulos 2 y 3 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de cargar una base preparada, revisar su estructura, calcular frecuencias, construir gráficas e interpretar patrones iniciales.

## Cargar paquetes y base


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Explicación del código
Se carga la base preparada en el capítulo anterior. Esto permite empezar directamente con el análisis exploratorio.

## Preparar variables


In [ ]:
if (!is.null(atus_ml)) {
  atus_ml <- atus_ml |>
    mutate(
      ID_HORA_NUM = as.numeric(ID_HORA),
      MES_NUM = as.numeric(MES),
      MES_FACTOR = factor(sprintf("%02d", MES_NUM), levels = sprintf("%02d", 1:12)),
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))
    )
}


## Explicación del código
Se crean variables numéricas y factores ordenados para facilitar tablas y gráficas.

## Distribución de la variable respuesta


In [ ]:
if (!is.null(atus_ml)) {
  tabla_clase <- table(atus_ml$accidente_con_victimas)
  round(100 * prop.table(tabla_clase), 2)
}

if (!is.null(atus_ml)) {
  ggplot(atus_ml, aes(x = accidente_con_victimas, fill = accidente_con_victimas)) +
    geom_bar(width = 0.7) +
    escala_clases_fill() +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Distribución de accidentes según presencia de víctimas",
      subtitle = "Comparación entre clases",
      x = "Clase",
      y = "Número de accidentes",
      fill = "Clase"
    ) +
    tema_libro() +
    theme(legend.position = "none")
}


## Interpretación del resultado
La base está desbalanceada: hay más accidentes de solo daños que accidentes con víctimas. Esto será importante al evaluar modelos.

## Accidentes por mes


In [ ]:
if (!is.null(atus_ml)) {
  accidentes_mes <- atus_ml |> count(MES_FACTOR, name = "n") |> arrange(MES_FACTOR)
  accidentes_mes
}

if (exists("accidentes_mes")) {
  ggplot(accidentes_mes, aes(x = MES_FACTOR, y = n)) +
    geom_col(fill = col_azul, width = 0.75) +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Número de accidentes por mes",
      subtitle = "Base ATUS preparada",
      x = "Mes",
      y = "Número de accidentes"
    ) +
    tema_libro()
}


## Explicación del código
Se cuentan los accidentes por mes y se visualizan con barras. El eje vertical usa separadores de miles para facilitar la lectura.

## Accidentes por hora del día


In [ ]:
if (!is.null(atus_ml)) {
  accidentes_hora <- atus_ml |> count(ID_HORA_NUM, name = "n") |> arrange(ID_HORA_NUM)

  ggplot(accidentes_hora, aes(x = ID_HORA_NUM, y = n)) +
    geom_col(fill = col_turquesa, width = 0.75) +
    scale_x_continuous(breaks = 0:23) +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Número de accidentes por hora del día",
      x = "Hora del día",
      y = "Número de accidentes"
    ) +
    tema_libro()
}


## Interpretación del resultado
La gráfica permite detectar horas con mayor concentración de accidentes.

## Tipos de accidente más frecuentes


In [ ]:
if (!is.null(atus_ml)) {
  accidentes_tipo_top <- atus_ml |> count(TIPACCID, name = "n") |> slice_max(n, n = 10)

  ggplot(accidentes_tipo_top, aes(x = reorder(TIPACCID, n), y = n)) +
    geom_col(fill = col_azul, width = 0.75) +
    coord_flip() +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Tipos de accidente más frecuentes",
      x = "Tipo de accidente",
      y = "Número de accidentes"
    ) +
    tema_libro()
}


## Explicación del código
Se seleccionan los diez tipos de accidente con mayor frecuencia para evitar una gráfica saturada.

## Proporción por tipo de accidente


In [ ]:
if (!is.null(atus_ml)) {
  tipos_top <- atus_ml |> count(TIPACCID) |> slice_max(n, n = 10) |> pull(TIPACCID)
  atus_tipo_top <- atus_ml |> filter(TIPACCID %in% tipos_top)

  ggplot(atus_tipo_top, aes(x = TIPACCID, fill = accidente_con_victimas)) +
    geom_bar(position = "fill") +
    coord_flip() +
    escala_clases_fill() +
    scale_y_continuous(labels = etiqueta_porcentaje) +
    labs(
      title = "Proporción de accidentes con víctimas según tipo de accidente",
      x = "Tipo de accidente",
      y = "Proporción",
      fill = "Clase"
    ) +
    tema_libro()
}


## Interpretación del resultado
Esta gráfica compara proporciones, no cantidades absolutas. Permite identificar tipos de accidente con mayor presencia relativa de víctimas.

## Materiales complementarios del capítulo
Estos recursos permiten repasar los conceptos esenciales del análisis exploratorio de datos mediante distintos formatos.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual del análisis exploratorio de datos y de sus principales herramientas. | [Ver en YouTube](https://www.youtube.com/watch?v=U2hd8iEovCY) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pdf) | [Descargar PDF](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pdf){download="capitulo-03-analisis-exploratorio-datos.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pptx) | [Descargar PPTX](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pptx){download="capitulo-03-analisis-exploratorio-datos.pptx"} |
| Infografía | Síntesis visual de conceptos, procedimientos y gráficos del capítulo. | [Ver infografía](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png) | [Descargar PNG](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png){download="capitulo-03-analisis-exploratorio-datos-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/03-analisis-exploratorio.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 3](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png)](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/03-analisis-exploratorio.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=U2hd8iEovCY>

La presentación PDF, el archivo editable y la infografía pueden descargarse desde la versión web del libro.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

## Laboratorio interactivo: exploración de distribuciones

Selecciona una variable de `iris` y examina su distribución.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

Permite elegir una variable y visualizar histograma, densidad o boxplot.

## Caso aplicado B: análisis exploratorio de COVID-19

La muestra de 2022 contiene 35 000 registros sin defunción y 15 000 con
defunción registrada. Esta proporción fue construida deliberadamente para
facilitar el aprendizaje de clasificación y **no representa la mortalidad
poblacional real**.


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

covid <- covid |>
  mutate(
    desenlace = factor(
      MURIO,
      levels = c(0, 1),
      labels = c("Sin defunción", "Defunción registrada")
    ),
    paciente = factor(
      TIPO_PACIENTE,
      levels = c(0, 1),
      labels = c("Ambulatorio", "Hospitalizado")
    )
  )


### Distribución de la edad


In [ ]:
ggplot(covid, aes(x = EDAD, fill = desenlace)) +
  geom_histogram(
    bins = 35,
    position = "identity",
    alpha = 0.55
  ) +
  facet_wrap(~ desenlace, ncol = 1) +
  labs(
    x = "Edad",
    y = "Frecuencia",
    fill = "Desenlace"
  ) +
  theme_minimal()


### Mortalidad observada por grupos de edad


In [ ]:
resumen_edad <- covid |>
  mutate(
    grupo_edad = cut(
      EDAD,
      breaks = c(-Inf, 19, 39, 59, 69, 79, Inf),
      labels = c(
        "0-19", "20-39", "40-59",
        "60-69", "70-79", "80 o más"
      )
    )
  ) |>
  group_by(grupo_edad) |>
  summarise(
    registros = n(),
    defunciones = sum(MURIO, na.rm = TRUE),
    proporcion_muestra = mean(MURIO, na.rm = TRUE),
    .groups = "drop"
  )

resumen_edad


### Comorbilidades y desenlace


In [ ]:
covid |>
  group_by(NUM_COMORBILIDADES) |>
  summarise(
    registros = n(),
    proporcion_defuncion_muestra = mean(MURIO, na.rm = TRUE),
    .groups = "drop"
  )


La proporción calculada corresponde a la muestra educativa balanceada. No debe
interpretarse como una tasa epidemiológica de mortalidad.

## Laboratorio interactivo ATUS: mes, hora y víctimas

Este laboratorio utiliza **conteos agregados calculados a partir de la base ATUS preparada del libro**. Permite seleccionar un mes y observar cómo cambia el número de accidentes por hora y la proporción de accidentes con víctimas.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio interactivo ATUS disponible en la versión web

La versión web permite seleccionar el mes, comparar accidentes por hora y observar el porcentaje de accidentes con víctimas usando agregados de la base ATUS preparada.

## Conclusión

El análisis exploratorio permite detectar patrones iniciales, revisar el desbalance de clases e identificar variables útiles para modelar.


# Regresión lineal simple y múltiple

La formulación matemática de **regresión lineal simple y múltiple** se desarrolla con mayor profundidad
en los capítulos 9 y 10 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al terminar este capítulo, el lector podrá:

- distinguir entre regresión y clasificación;
- interpretar una relación lineal entre una variable respuesta y una variable explicativa;
- ajustar una regresión lineal simple en R;
- construir e interpretar un modelo de regresión lineal múltiple;
- evaluar el ajuste mediante $R^2$, $R^2$ ajustado, RMSE y análisis de residuos;
- reconocer cuándo la regresión lineal no es apropiada.

## ¿Por qué estudiar regresión lineal en un libro de clasificación?

La regresión lineal no es un algoritmo de clasificación: su respuesta es numérica y continua. Sin embargo, es una base fundamental del aprendizaje supervisado porque introduce ideas que reaparecen en otros modelos: función de predicción, coeficientes, error, ajuste, generalización y evaluación fuera de muestra.

Además, permite comprender mejor la regresión logística del capítulo siguiente. Ambos modelos construyen una combinación de variables explicativas, aunque difieren en la naturaleza de la respuesta y en la función utilizada para obtener la predicción [@james2021islr].

En regresión lineal se predice una cantidad, por ejemplo el número mensual de accidentes. En clasificación se predice una categoría, por ejemplo si un accidente tuvo víctimas o solo daños.

## Regresión lineal simple

La regresión lineal simple relaciona una variable respuesta $Y$ con una sola variable explicativa $X$:

$$
Y_i = \beta_0 + \beta_1 X_i + \varepsilon_i.
$$

Donde:

- $\beta_0$ es la ordenada al origen;
- $\beta_1$ representa el cambio promedio esperado en $Y$ cuando $X$ aumenta una unidad;
- $\varepsilon_i$ representa la parte no explicada por el modelo.

Los coeficientes se estiman normalmente mediante mínimos cuadrados, buscando minimizar la suma de los errores al cuadrado [@montgomery2021introduction].

## Primer ejemplo didáctico

Supongamos que se desea relacionar el número de horas de estudio con la calificación obtenida.


In [ ]:
horas <- c(2, 3, 4, 5, 6, 7, 8, 9)
calificacion <- c(55, 60, 64, 70, 75, 79, 86, 90)

datos_estudio <- data.frame(
  horas = horas,
  calificacion = calificacion
)

datos_estudio


## Visualizar la relación


In [ ]:
library(ggplot2)
source("util_graficas.R")

ggplot(datos_estudio, aes(x = horas, y = calificacion)) +
  geom_point(size = 3) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "Horas de estudio y calificación",
    subtitle = "Ejemplo de una relación aproximadamente lineal",
    x = "Horas de estudio",
    y = "Calificación"
  ) +
  tema_libro()


## Ajustar el modelo simple


In [ ]:
modelo_simple <- lm(
  calificacion ~ horas,
  data = datos_estudio
)

summary(modelo_simple)


## Interpretar los coeficientes


In [ ]:
coeficientes_simple <- coef(modelo_simple)
coeficientes_simple


El modelo estimado tiene la forma:

$$
\widehat{Y} = \widehat{\beta}_0 + \widehat{\beta}_1 X.
$$

La pendiente indica cuántos puntos cambia, en promedio, la calificación por cada hora adicional de estudio. La ordenada al origen es necesaria matemáticamente, aunque no siempre tenga una interpretación práctica dentro del rango observado.

## Realizar una predicción


In [ ]:
nuevo_estudiante <- data.frame(horas = 6.5)

predict(
  modelo_simple,
  newdata = nuevo_estudiante,
  interval = "prediction",
  level = 0.95
)


El intervalo de predicción es más amplio que un intervalo para la media porque considera tanto la incertidumbre del modelo como la variabilidad individual.

## Residuos

El residuo de la observación $i$ es:

$$
e_i = y_i - \widehat{y}_i.
$$


In [ ]:
diagnostico_simple <- data.frame(
  observado = datos_estudio$calificacion,
  predicho = fitted(modelo_simple),
  residuo = residuals(modelo_simple)
)

diagnostico_simple


## Visualizar los residuos


In [ ]:
ggplot(diagnostico_simple, aes(x = predicho, y = residuo)) +
  geom_hline(yintercept = 0, linetype = 2) +
  geom_point(size = 3) +
  labs(
    title = "Residuos frente a valores predichos",
    x = "Valor predicho",
    y = "Residuo"
  ) +
  tema_libro()


Una nube sin patrón claro alrededor de cero es compatible con una relación lineal razonable. Curvaturas, forma de embudo o puntos extremos sugieren revisar el modelo.

## Regresión lineal múltiple

La regresión múltiple incorpora varias variables explicativas:

$$
Y_i = \beta_0 + \beta_1X_{i1} + \beta_2X_{i2} + \cdots + \beta_pX_{ip} + \varepsilon_i.
$$

Cada coeficiente representa el cambio esperado en la respuesta cuando la variable correspondiente aumenta una unidad, **manteniendo constantes las demás variables**.

## Preparar un ejemplo múltiple


In [ ]:
set.seed(2026)

n <- 80

datos_ambientales <- data.frame(
  temperatura = runif(n, 12, 35),
  velocidad_viento = runif(n, 0.5, 8),
  trafico = runif(n, 100, 900)
)

datos_ambientales$pm10 <-
  18 +
  0.9 * datos_ambientales$temperatura -
  2.1 * datos_ambientales$velocidad_viento +
  0.045 * datos_ambientales$trafico +
  rnorm(n, mean = 0, sd = 6)

head(datos_ambientales)


## Ajustar el modelo múltiple


In [ ]:
modelo_multiple <- lm(
  pm10 ~ temperatura + velocidad_viento + trafico,
  data = datos_ambientales
)

summary(modelo_multiple)


## Interpretar los coeficientes múltiples


In [ ]:
tabla_coeficientes <- data.frame(
  termino = names(coef(modelo_multiple)),
  estimacion = as.numeric(coef(modelo_multiple))
)

tabla_coeficientes


Por ejemplo, el coeficiente de `velocidad_viento` estima el cambio promedio de PM10 asociado con un aumento de una unidad en la velocidad del viento, manteniendo constantes la temperatura y el tráfico.

Un coeficiente estadísticamente significativo no demuestra por sí solo una relación causal. La causalidad requiere diseño, conocimiento sustantivo y control adecuado de variables de confusión.

## Separar entrenamiento y prueba

Evaluar el modelo con los mismos datos usados para ajustarlo produce una visión demasiado optimista. Por ello se separan datos de entrenamiento y prueba.


In [ ]:
set.seed(2026)

indices_entrenamiento <- sample(
  seq_len(nrow(datos_ambientales)),
  size = floor(0.8 * nrow(datos_ambientales))
)

entrenamiento_reg <- datos_ambientales[indices_entrenamiento, ]
prueba_reg <- datos_ambientales[-indices_entrenamiento, ]

modelo_multiple_prueba <- lm(
  pm10 ~ temperatura + velocidad_viento + trafico,
  data = entrenamiento_reg
)

prediccion_reg <- predict(
  modelo_multiple_prueba,
  newdata = prueba_reg
)


## Métricas de evaluación

El error cuadrático medio y su raíz se calculan como:

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat{y}_i)^2}.
$$


In [ ]:
rmse <- sqrt(mean((prueba_reg$pm10 - prediccion_reg)^2))
mae <- mean(abs(prueba_reg$pm10 - prediccion_reg))

metricas_regresion <- data.frame(
  metrica = c("RMSE", "MAE"),
  valor = c(rmse, mae)
)

metricas_regresion


## Observado frente a predicho


In [ ]:
comparacion_regresion <- data.frame(
  observado = prueba_reg$pm10,
  predicho = prediccion_reg
)

ggplot(comparacion_regresion, aes(x = observado, y = predicho)) +
  geom_point(size = 3, alpha = 0.8) +
  geom_abline(slope = 1, intercept = 0, linetype = 2) +
  labs(
    title = "Valores observados y predichos",
    subtitle = "La línea diagonal representa predicción perfecta",
    x = "PM10 observado",
    y = "PM10 predicho"
  ) +
  tema_libro()


## Aplicación con datos ATUS agregados

En ATUS cada fila corresponde a un accidente. La base preparada del libro conserva las variables necesarias para los algoritmos de clasificación, pero no incluye entidad ni municipio. Por ello, en este ejemplo los registros se agregan por mes y se modela el número mensual de accidentes registrados.


In [ ]:
library(readr)
library(dplyr)

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (!file.exists(ruta_atus_ml)) {
  stop(
    paste(
      "No se encontró datos/atus_ml_preparado.csv.",
      "Ejecute primero la celda de preparación automática."
    )
  )
}

atus_reg <- read_csv(
  ruta_atus_ml,
  show_col_types = FALSE
)

names(atus_reg)


## Construir una base mensual


In [ ]:
variables_necesarias <- c(
  "MES", "ID_HORA", "DIASEMANA", "TIPACCID",
  "accidente_con_victimas"
)

faltantes <- setdiff(variables_necesarias, names(atus_reg))

if (length(faltantes) > 0) {
  stop(
    paste(
      "Faltan variables para el ejemplo:",
      paste(faltantes, collapse = ", ")
    )
  )
}

atus_mensual <- atus_reg |>
  mutate(
    ID_HORA = as.numeric(ID_HORA),
    MES = as.numeric(MES),
    fin_semana = DIASEMANA %in% c(
      "sábado", "sabado", "domingo",
      "Sábado", "Sabado", "Domingo"
    ),
    con_victimas = accidente_con_victimas == "Con víctimas"
  ) |>
  group_by(MES) |>
  summarise(
    accidentes = n(),
    tipos_accidente = n_distinct(TIPACCID, na.rm = TRUE),
    hora_promedio = mean(ID_HORA, na.rm = TRUE),
    proporcion_fin_semana = mean(fin_semana, na.rm = TRUE),
    proporcion_con_victimas = mean(con_victimas, na.rm = TRUE),
    .groups = "drop"
  ) |>
  filter(
    is.finite(accidentes),
    is.finite(tipos_accidente),
    is.finite(hora_promedio),
    is.finite(proporcion_fin_semana),
    is.finite(proporcion_con_victimas)
  )

atus_mensual


## Ajustar la regresión múltiple con ATUS


In [ ]:
modelo_atus_reg <- lm(
  accidentes ~ MES + tipos_accidente +
    hora_promedio + proporcion_fin_semana,
  data = atus_mensual
)

summary(modelo_atus_reg)


El número de accidentes es una variable de conteo. La regresión lineal se utiliza aquí con fines didácticos; en un estudio formal podrían ser más apropiados modelos de Poisson o binomial negativa. Esta comparación ayuda a comprender que la elección del algoritmo depende de la naturaleza de la respuesta.

## Supuestos principales

La interpretación inferencial tradicional de la regresión lineal se apoya en varios supuestos:

1. relación aproximadamente lineal;
2. errores independientes;
3. varianza aproximadamente constante;
4. ausencia de colinealidad extrema;
5. normalidad aproximada de los errores cuando se construyen pruebas e intervalos.

Los supuestos deben analizarse mediante gráficas, conocimiento del problema y medidas diagnósticas; no deben tratarse como una lista mecánica.

## Diferencias entre regresión lineal y logística

| Característica | Regresión lineal | Regresión logística |
|---|---|---|
| Respuesta | Numérica continua | Categórica binaria |
| Salida directa | Cualquier número real | Probabilidad entre 0 y 1 |
| Función usual | Identidad | Logística |
| Evaluación | RMSE, MAE, $R^2$ | Matriz de confusión, sensibilidad, especificidad, AUC |
| Ejemplo | Predecir PM10 | Clasificar accidente con víctimas |

## Laboratorio interactivo: regresión lineal

Modifica el intercepto, la pendiente y el nivel de ruido para observar cómo cambia la nube de puntos y la recta de regresión.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El laboratorio permite modificar intercepto, pendiente, ruido, tamaño de muestra y semilla, y comparar la recta verdadera con la estimada.

## Actividades para el lector

1. Modifique el ejemplo simple y prediga la calificación para 7.5 horas de estudio.
2. Retire una variable del modelo ambiental y compare el $R^2$ ajustado y el RMSE.
3. Añada una interacción entre temperatura y velocidad del viento.
4. Analice los residuos del modelo múltiple.
5. Explique por qué no sería correcto utilizar regresión lineal ordinaria para predecir directamente una categoría como `Con víctimas` o `Solo daños`.
6. Compare conceptualmente el modelo ATUS de este capítulo con la regresión logística del capítulo siguiente.

## Conclusiones

La regresión lineal simple explica una respuesta cuantitativa mediante una variable y la regresión múltiple permite considerar varios factores simultáneamente. Su valor en este libro no se limita a la predicción numérica: proporciona el lenguaje básico para comprender coeficientes, errores, validación y generalización, conceptos que continuarán apareciendo en los algoritmos de clasificación.

## Materiales complementarios del capítulo
| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de la regresión lineal simple y múltiple. | [Ver en YouTube](https://www.youtube.com/watch?v=SE6DCeU9h1w) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pdf) | [Descargar PDF](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pdf){download="capitulo-04-regresion-lineal-simple-multiple.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pptx) | [Descargar PPTX](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple.pptx){download="capitulo-04-regresion-lineal-simple-multiple.pptx"} |
| Infografía | Síntesis visual de conceptos, fórmulas e interpretación. | [Ver infografía](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png) | [Descargar PNG](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png){download="capitulo-04-regresion-lineal-simple-multiple-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/04-regresion-lineal.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 4](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png)](recursos/capitulo-04/capitulo-04-regresion-lineal-simple-multiple-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/04-regresion-lineal.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=SE6DCeU9h1w>

**Curso completo en YouTube:** <https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q>

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)


# Regresión logística para clasificación

La formulación matemática de **regresión logística, máxima verosimilitud y clasificación probabilística** se desarrolla con mayor profundidad
en los capítulos 3, 6 y 11 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de ajustar una regresión logística, predecir probabilidades y evaluar el desempeño.

## Fundamento matemático

La regresión logística utiliza la función:

$$
p = \frac{1}{1 + e^{-z}}
$$

donde $z = \beta_0 + \beta_1x_1 + \cdots + \beta_px_p$.

## Visualización de la función logística


In [ ]:
library(ggplot2)
source("util_graficas.R")

z <- seq(-10, 10, length.out = 200)
datos_logistica <- data.frame(z = z, p = 1 / (1 + exp(-z)))

ggplot(datos_logistica, aes(x = z, y = p)) +
  geom_line(linewidth = 1.2, color = col_azul) +
  labs(
    title = "Función logística",
    subtitle = "Transforma valores reales en probabilidades",
    x = "z",
    y = "Probabilidad"
  ) +
  tema_libro()


## Explicación del código
Se genera una secuencia de valores para $z$ y se calcula su probabilidad usando la función logística.

## Interpretación del resultado
La curva tiene forma de S. Valores negativos producen probabilidades cercanas a cero y valores positivos probabilidades cercanas a uno.

## Cargar paquetes y datos


In [ ]:
library(readr)
library(dplyr)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Preparar y dividir datos


In [ ]:
if (!is.null(atus_ml)) {
  atus_modelo <- atus_ml |>
    mutate(
      victimas_binaria = ifelse(accidente_con_victimas == "Con víctimas", 1, 0),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI),
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))
    )

  set.seed(123)
  idx <- sample(1:nrow(atus_modelo), size = round(0.7 * nrow(atus_modelo)))
  entrenamiento <- atus_modelo[idx, ]
  prueba <- atus_modelo[-idx, ]
}


## Explicación del código
La variable respuesta se transforma a binaria y la base se divide en entrenamiento y prueba.

## Ajustar modelo


In [ ]:
if (exists("entrenamiento")) {
  modelo_logistico <- glm(
    victimas_binaria ~ MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = entrenamiento,
    family = binomial
  )

  head(summary(modelo_logistico)$coefficients, 12)
}


## Interpretación del resultado
Los coeficientes estiman el efecto de cada variable sobre el logit de la probabilidad de accidente con víctimas.

## Predicción y matriz de confusión


In [ ]:
if (exists("modelo_logistico")) {
  prueba$probabilidad_victimas <- predict(modelo_logistico, newdata = prueba, type = "response")
  prueba$prediccion_03 <- factor(
    ifelse(prueba$probabilidad_victimas >= 0.3, "Con víctimas", "Solo daños"),
    levels = c("Con víctimas", "Solo daños")
  )

  matriz_confusion_03 <- table(
    Real = prueba$accidente_con_victimas,
    Predicho = prueba$prediccion_03
  )

  matriz_confusion_03
}


## Explicación del código
Las probabilidades se convierten en clases usando un punto de corte de 0.3.

## Métricas


In [ ]:
if (exists("matriz_confusion_03")) {
  VP <- matriz_confusion_03["Con víctimas", "Con víctimas"]
  FN <- matriz_confusion_03["Con víctimas", "Solo daños"]
  FP <- matriz_confusion_03["Solo daños", "Con víctimas"]
  VN <- matriz_confusion_03["Solo daños", "Solo daños"]

  data.frame(
    exactitud = (VP + VN) / (VP + FN + FP + VN),
    sensibilidad = VP / (VP + FN),
    especificidad = VN / (VN + FP)
  )
}


## Interpretación del resultado
La sensibilidad mide la capacidad para detectar accidentes con víctimas; la especificidad mide la capacidad para detectar accidentes de solo daños.

## Distribución de probabilidades


In [ ]:
if (exists("prueba")) {
  ggplot(prueba, aes(x = probabilidad_victimas, fill = accidente_con_victimas)) +
    geom_histogram(bins = 30, alpha = 0.75, position = "identity") +
    escala_clases_fill(name = "Clase real") +
    labs(
      title = "Distribución de probabilidades predichas",
      subtitle = "Regresión logística",
      x = "Probabilidad predicha de accidente con víctimas",
      y = "Frecuencia"
    ) +
    tema_libro()
}


## Materiales complementarios del capítulo
Estos recursos permiten estudiar la regresión logística desde una perspectiva audiovisual, gráfica y práctica.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de la regresión logística aplicada a problemas de clasificación. | [Ver en YouTube](https://www.youtube.com/watch?v=7CwP40QDeys) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pdf) | [Descargar PDF](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pdf){download="capitulo-05-regresion-logistica-clasificacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pptx) | [Descargar PPTX](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion.pptx){download="capitulo-05-regresion-logistica-clasificacion.pptx"} |
| Infografía | Síntesis visual de los conceptos, la función logística y el proceso de clasificación. | [Ver infografía](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png) | [Descargar PNG](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png){download="capitulo-05-regresion-logistica-clasificacion-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/05-regresion-logistica.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 5](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png)](recursos/capitulo-05/capitulo-05-regresion-logistica-clasificacion-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/05-regresion-logistica.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=7CwP40QDeys>

**Curso completo en YouTube:** <https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q>

La presentación PDF, el archivo PowerPoint y la infografía pueden descargarse desde la versión web del libro.

Este video forma parte de la lista oficial del curso.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor.

## Laboratorio interactivo: regresión logística

Explora cómo el intercepto, la pendiente y el umbral cambian la probabilidad estimada y la clasificación final.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El laboratorio permite cambiar los coeficientes, el umbral y el valor de una observación nueva para estudiar la curva logística y la clase predicha.

## Caso aplicado B: regresión logística con COVID-19

Construiremos un modelo para estimar la probabilidad de una **defunción
registrada** a partir de variables demográficas y clínicas.


In [ ]:
library(readr)
library(dplyr)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

covid_modelo <- covid |>
  select(
    MURIO,
    EDAD,
    NEUMONIA,
    DIABETES,
    HIPERTENSION,
    OBESIDAD,
    RENAL_CRONICA
  ) |>
  tidyr::drop_na()

set.seed(2026)

indice_entrenamiento <- sample(
  seq_len(nrow(covid_modelo)),
  size = floor(0.80 * nrow(covid_modelo))
)

covid_train <- covid_modelo[indice_entrenamiento, ]
covid_test <- covid_modelo[-indice_entrenamiento, ]

modelo_covid <- glm(
  MURIO ~ EDAD + NEUMONIA + DIABETES +
    HIPERTENSION + OBESIDAD + RENAL_CRONICA,
  data = covid_train,
  family = binomial()
)

summary(modelo_covid)


### Razones de momios


In [ ]:
odds_ratios <- data.frame(
  variable = names(coef(modelo_covid)),
  coeficiente = coef(modelo_covid),
  odds_ratio = exp(coef(modelo_covid))
)

odds_ratios


Un `odds_ratio` mayor que uno indica que, manteniendo constantes las demás
variables, el predictor se asocia con un aumento de los momios del desenlace.
Esto **no demuestra causalidad**.

### Probabilidades para la base de prueba


In [ ]:
covid_test$probabilidad <- predict(
  modelo_covid,
  newdata = covid_test,
  type = "response"
)

head(
  covid_test |>
    arrange(desc(probabilidad)),
  10
)


## Laboratorio interactivo: probabilidad estimada COVID-19

El simulador usa coeficientes estimados previamente con la muestra educativa.
Su propósito es mostrar el funcionamiento matemático de una regresión
logística; **no es una calculadora clínica ni un instrumento de diagnóstico**.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El lector puede modificar edad, neumonía y comorbilidades, además del umbral
de clasificación, para observar cómo cambia la probabilidad estimada.

## Conclusión

La regresión logística es un modelo interpretable para clasificación binaria. El punto de corte permite ajustar el balance entre sensibilidad y especificidad.

## Referencias fundamentales de la regresión logística

La formulación moderna de la regresión para respuestas binarias se relaciona con el trabajo de Cox [@cox1958regression]. Para profundizar en ajuste, interpretación, diagnóstico y aplicaciones del modelo logístico puede consultarse a @hosmer2013applied. Una presentación orientada al aprendizaje estadístico y a su implementación en R aparece en @james2021islr.


# k vecinos más cercanos, k-NN

La formulación matemática de **distancias, escalamiento y vecinos más cercanos** se desarrolla con mayor profundidad
en los capítulos 2 y 12 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar k-NN, preparar datos para este método, estandarizar variables y evaluar el desempeño.

## Idea intuitiva

k-NN clasifica una nueva observación según la clase más común entre sus vecinos más cercanos.


In [ ]:
library(ggplot2)
library(class)
library(dplyr)
library(tidyr)
library(readr)
source("util_graficas.R")

datos_ejemplo <- data.frame(
  hora = c(8, 9, 10, 18, 19, 20, 22, 23),
  mes = c(1, 1, 2, 6, 6, 7, 12, 12),
  clase = c("Solo daños", "Solo daños", "Solo daños", "Con víctimas", "Con víctimas", "Con víctimas", "Con víctimas", "Con víctimas")
)

ggplot(datos_ejemplo, aes(x = hora, y = mes, color = clase)) +
  geom_point(size = 4) +
  escala_clases_color() +
  labs(title = "Ejemplo intuitivo de k-NN", x = "Hora", y = "Mes", color = "Clase") +
  tema_libro()


## Explicación del código
Se construyen puntos artificiales para visualizar la idea de vecinos cercanos.

## Distancia euclidiana

$$
d(A,B) = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}
$$


In [ ]:
sqrt((8 - 10)^2 + (1 - 2)^2)


## Interpretación del resultado
Una distancia menor indica mayor similitud entre dos observaciones.

## Cargar y preparar datos


In [ ]:
ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  set.seed(123)
  atus_knn_base <- atus_ml |>
    sample_n(min(12000, nrow(atus_ml))) |>
    mutate(
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños")),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI)
    ) |>
    na.omit()
}


## Explicación del código
Se toma una muestra para que k-NN sea rápido y reproducible. Este método puede ser costoso en bases grandes.

## Modelo k-NN


In [ ]:
if (exists("atus_knn_base")) {
  matriz_predictoras <- model.matrix(~ ID_HORA + MES + DIASEMANA + TIPACCID + CAUSAACCI - 1, data = atus_knn_base)
  y <- atus_knn_base$accidente_con_victimas

  set.seed(123)
  idx <- sample(1:nrow(matriz_predictoras), size = round(0.7 * nrow(matriz_predictoras)))
  x_entrenamiento <- matriz_predictoras[idx, ]
  x_prueba <- matriz_predictoras[-idx, ]
  y_entrenamiento <- y[idx]
  y_prueba <- y[-idx]

  medias <- apply(x_entrenamiento, 2, mean)
  desviaciones <- apply(x_entrenamiento, 2, sd)
  desviaciones[desviaciones == 0] <- 1

  x_entrenamiento_esc <- scale(x_entrenamiento, center = medias, scale = desviaciones)
  x_prueba_esc <- scale(x_prueba, center = medias, scale = desviaciones)

  pred_knn_5 <- knn(x_entrenamiento_esc, x_prueba_esc, y_entrenamiento, k = 5)
  table(Real = y_prueba, Predicho = pred_knn_5)
}


## Explicación del código
Se crean variables dummy, se estandarizan las columnas y se clasifica con k igual a 5.

## Comparación de valores de k


In [ ]:
calcular_metricas <- function(real, predicho) {
  real <- factor(real, levels = c("Con víctimas", "Solo daños"))
  predicho <- factor(predicho, levels = c("Con víctimas", "Solo daños"))
  m <- table(Real = real, Predicho = predicho)
  VP <- m["Con víctimas", "Con víctimas"]
  FN <- m["Con víctimas", "Solo daños"]
  FP <- m["Solo daños", "Con víctimas"]
  VN <- m["Solo daños", "Solo daños"]
  data.frame(exactitud=(VP+VN)/(VP+FN+FP+VN), sensibilidad=VP/(VP+FN), especificidad=VN/(VN+FP))
}

if (exists("x_entrenamiento_esc")) {
  valores_k <- c(3, 5, 11)
  resultados_knn <- data.frame()

  for (k_actual in valores_k) {
    pred_actual <- knn(x_entrenamiento_esc, x_prueba_esc, y_entrenamiento, k = k_actual)
    tmp <- calcular_metricas(y_prueba, pred_actual)
    tmp$k <- k_actual
    resultados_knn <- rbind(resultados_knn, tmp)
  }

  resultados_knn <- resultados_knn |> select(k, exactitud, sensibilidad, especificidad)
  resultados_knn
}

if (exists("resultados_knn")) {
  resultados_largos <- resultados_knn |>
    pivot_longer(cols = c(exactitud, sensibilidad, especificidad), names_to = "metrica", values_to = "valor")

  ggplot(resultados_largos, aes(x = k, y = valor, color = metrica, group = metrica)) +
    geom_line(linewidth = 1.1) +
    geom_point(size = 3) +
    scale_color_manual(values = c(exactitud = col_azul, sensibilidad = col_coral, especificidad = col_turquesa)) +
    labs(title = "Comparación de métricas para distintos valores de k", x = "Valor de k", y = "Valor") +
    tema_libro()
}


## Interpretación del resultado
Cambiar k modifica el balance entre exactitud, sensibilidad y especificidad. No existe un k universalmente mejor.

## Laboratorio interactivo: k-NN

Selecciona el número de vecinos y mueve un punto nuevo. El laboratorio identifica sus vecinos más cercanos y muestra la clase predicha.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El lector puede cambiar el valor de $k$, mover una observación nueva y visualizar sus vecinos, distancias y clase predicha.

## Caso aplicado B: k-NN con COVID-19

En esta segunda ruta aplicada usamos la misma muestra educativa de COVID-19 México 2022 empleada en los capítulos anteriores. El objetivo es clasificar `MURIO` a partir de edad, neumonía, diabetes, hipertensión, obesidad, enfermedad renal crónica y número de comorbilidades.

> **Uso académico:** este ejercicio permite estudiar el comportamiento de k-NN. No es una calculadora clínica ni un instrumento de diagnóstico.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_knn <- read_csv(ruta_covid, show_col_types = FALSE) |>
    select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
           OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    drop_na()

  set.seed(2026)
  covid_knn <- covid_knn |>
    sample_n(min(12000, nrow(covid_knn))) |>
    mutate(
      MURIO = factor(MURIO, levels = c(1, 0), labels = c("Defunción", "Sin defunción"))
    )

  X_covid <- covid_knn |> select(-MURIO) |> as.matrix()
  y_covid <- covid_knn$MURIO

  set.seed(2026)
  idx_covid <- sample(seq_len(nrow(X_covid)), size = floor(0.80 * nrow(X_covid)))

  x_train_covid <- X_covid[idx_covid, , drop = FALSE]
  x_test_covid <- X_covid[-idx_covid, , drop = FALSE]
  y_train_covid <- y_covid[idx_covid]
  y_test_covid <- y_covid[-idx_covid]

  medias_covid <- apply(x_train_covid, 2, mean)
  desv_covid <- apply(x_train_covid, 2, sd)
  desv_covid[desv_covid == 0] <- 1

  x_train_covid_z <- scale(x_train_covid, center = medias_covid, scale = desv_covid)
  x_test_covid_z <- scale(x_test_covid, center = medias_covid, scale = desv_covid)
}


La estandarización es especialmente importante en k-NN porque la distancia euclidiana sería dominada por variables con escalas mayores, como la edad.


In [ ]:
if (exists("x_train_covid_z")) {
  pred_covid_knn <- knn(train = x_train_covid_z, test = x_test_covid_z, cl = y_train_covid, k = 11)

  matriz_covid_knn <- table(
    Real = factor(y_test_covid, levels = c("Defunción", "Sin defunción")),
    Predicho = factor(pred_covid_knn, levels = c("Defunción", "Sin defunción"))
  )
  matriz_covid_knn
}


In [ ]:
if (exists("x_train_covid_z")) {
  evaluar_knn_covid <- function(k) {
    p <- knn(x_train_covid_z, x_test_covid_z, y_train_covid, k = k)
    m <- table(
      Real = factor(y_test_covid, levels = c("Defunción", "Sin defunción")),
      Predicho = factor(p, levels = c("Defunción", "Sin defunción"))
    )
    VP <- m[1,1]; FN <- m[1,2]; FP <- m[2,1]; VN <- m[2,2]
    data.frame(
      k = k,
      exactitud = (VP + VN) / sum(m),
      sensibilidad = ifelse(VP + FN == 0, NA, VP / (VP + FN)),
      especificidad = ifelse(VN + FP == 0, NA, VN / (VN + FP))
    )
  }

  resultados_k_covid <- do.call(rbind, lapply(c(3, 5, 11, 21), evaluar_knn_covid))
  resultados_k_covid
}


## Interpretación
El valor de $k$ controla cuánto se suaviza la decisión. Valores pequeños reaccionan más a observaciones locales; valores mayores producen fronteras más estables. En datos desbalanceados conviene mirar sensibilidad y especificidad, no solo exactitud.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=NLJj0QAWomU) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-06/capitulo-06-knn-presentacion.pdf) | [Descargar PDF](recursos/capitulo-06/capitulo-06-knn-presentacion.pdf){download="capitulo-06-knn-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-06/capitulo-06-knn-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-06/capitulo-06-knn-presentacion.pptx){download="capitulo-06-knn-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-06/capitulo-06-knn-infografia.png) | [Descargar PNG](recursos/capitulo-06/capitulo-06-knn-infografia.png){download="capitulo-06-knn-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/06-knn.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 6](recursos/capitulo-06/capitulo-06-knn-infografia.png)](recursos/capitulo-06/capitulo-06-knn-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/06-knn.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=NLJj0QAWomU>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

k-NN es un método intuitivo basado en similitud. Su desempeño depende del valor de $k$, del escalamiento y de la calidad de las variables predictoras.

## Referencias fundamentales de k-NN

El método de los vecinos más cercanos tiene como antecedente clásico el informe de @fix1951discriminatory. Sus propiedades de clasificación y cotas de error fueron estudiadas por @cover1967nearest. Una exposición moderna, acompañada de aplicaciones en R, puede consultarse en @james2021islr.


# Árboles de decisión para clasificación

La formulación matemática de **entropía, impureza, particiones y árboles de decisión** se desarrolla con mayor profundidad
en los capítulos 3, 6 y 13 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar la idea básica de un árbol de decisión, entrenarlo en R y evaluar su desempeño.

## Introducción

Los árboles de decisión clasifican observaciones mediante una secuencia de preguntas. Son interpretables y pueden expresarse como reglas tipo si-entonces.

## Fundamento matemático

Una medida común de impureza es el índice de Gini:

$$
Gini = 1 - \sum_{k=1}^{K} p_k^2
$$

## Cargar paquetes y datos


In [ ]:
library(readr)
library(dplyr)
library(rpart)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Usar la base completa


In [ ]:
if (!is.null(atus_ml)) {
  atus_arbol_base <- atus_ml |>
    mutate(
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños")),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI)
    ) |>
    na.omit()

  data.frame(filas = nrow(atus_arbol_base), columnas = ncol(atus_arbol_base))
}


## Explicación del código
Se usa la base completa, pero el crecimiento del árbol se controla mediante parámetros del modelo.

## División y entrenamiento


In [ ]:
if (exists("atus_arbol_base")) {
  set.seed(123)
  idx <- sample(1:nrow(atus_arbol_base), size = round(0.7 * nrow(atus_arbol_base)))
  entrenamiento <- atus_arbol_base[idx, ]
  prueba <- atus_arbol_base[-idx, ]

  modelo_arbol <- rpart(
    accidente_con_victimas ~ MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = entrenamiento,
    method = "class",
    control = rpart.control(cp = 0.005, minsplit = 1000, minbucket = 300, maxdepth = 5)
  )
}


## Explicación del código
Se ajusta un árbol de decisión controlado para evitar sobreajuste y salidas demasiado grandes.

## Complejidad e importancia


In [ ]:
if (exists("modelo_arbol")) printcp(modelo_arbol)

if (exists("modelo_arbol")) {
  importancia <- modelo_arbol$variable.importance
  if (!is.null(importancia)) {
    importancia_df <- data.frame(variable = names(importancia), importancia = as.numeric(importancia), row.names = NULL)
    importancia_df <- importancia_df[order(-importancia_df$importancia), ]
    importancia_df
  }
}


## Interpretación del resultado
La importancia de variables indica qué variables ayudan más a separar accidentes con víctimas de accidentes de solo daños.

## Gráfica opcional del árbol

La gráfica completa del árbol puede ser pesada en PDF. El lector puede ejecutarla manualmente en RStudio.


In [ ]:
plot(modelo_arbol, uniform = TRUE, margin = 0.1)
text(modelo_arbol, use.n = TRUE, cex = 0.7)


## Predicción y métricas


In [ ]:
if (exists("modelo_arbol")) {
  pred_arbol <- predict(modelo_arbol, newdata = prueba, type = "class")
  real_arbol <- factor(prueba$accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))
  pred_arbol <- factor(pred_arbol, levels = c("Con víctimas", "Solo daños"))

  matriz_confusion_arbol <- table(Real = real_arbol, Predicho = pred_arbol)
  matriz_confusion_arbol
}

if (exists("matriz_confusion_arbol")) {
  VP <- matriz_confusion_arbol["Con víctimas", "Con víctimas"]
  FN <- matriz_confusion_arbol["Con víctimas", "Solo daños"]
  FP <- matriz_confusion_arbol["Solo daños", "Con víctimas"]
  VN <- matriz_confusion_arbol["Solo daños", "Solo daños"]

  data.frame(exactitud=(VP+VN)/(VP+FN+FP+VN), sensibilidad=VP/(VP+FN), especificidad=VN/(VN+FP))
}


## Interpretación del resultado
La matriz de confusión y las métricas permiten evaluar el desempeño del árbol sobre datos de prueba.

## Laboratorio interactivo: construye una división del árbol

Este laboratorio permite explorar cómo una regla del tipo
`Petal.Length < punto de corte` divide los datos de `iris`. El usuario puede
cambiar el punto de corte y comparar la impureza de Gini antes y después de la
división.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

La versión web permite cambiar la variable y el punto de corte, observar la
división de las especies de `iris` y comparar la impureza de Gini de los nodos.

## Caso aplicado B: árbol de decisión con COVID-19

Usamos la misma muestra común de hasta 12 000 registros de COVID-19 México 2022 y la misma semilla `2026`. El árbol permite observar reglas de decisión fáciles de interpretar.

> **Uso académico:** las reglas encontradas describen el comportamiento de esta muestra y no deben interpretarse como reglas clínicas.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_arbol <- read_csv(ruta_covid, show_col_types = FALSE) |>
    select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
           OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_arbol <- covid_arbol |>
    sample_n(min(12000, nrow(covid_arbol))) |>
    mutate(MURIO = factor(MURIO, levels = c(1, 0), labels = c("Defunción", "Sin defunción")))

  set.seed(2026)
  idx_covid <- sample(seq_len(nrow(covid_arbol)), size = floor(0.80 * nrow(covid_arbol)))
  covid_train <- covid_arbol[idx_covid, ]
  covid_test <- covid_arbol[-idx_covid, ]
}


In [ ]:
if (exists("covid_train")) {
  arbol_covid <- rpart(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION + OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = covid_train,
    method = "class",
    control = rpart.control(cp = 0.002, minsplit = 80, minbucket = 30, maxdepth = 5)
  )
  printcp(arbol_covid)
}


In [ ]:
if (exists("arbol_covid")) {
  imp_covid_arbol <- arbol_covid$variable.importance
  if (!is.null(imp_covid_arbol)) {
    data.frame(variable = names(imp_covid_arbol), importancia = as.numeric(imp_covid_arbol)) |>
      arrange(desc(importancia))
  }
}


In [ ]:
if (exists("arbol_covid")) {
  pred_covid_arbol <- predict(arbol_covid, covid_test, type = "class")
  matriz_covid_arbol <- table(
    Real = factor(covid_test$MURIO, levels = c("Defunción", "Sin defunción")),
    Predicho = factor(pred_covid_arbol, levels = c("Defunción", "Sin defunción"))
  )
  matriz_covid_arbol

  VP <- matriz_covid_arbol[1,1]; FN <- matriz_covid_arbol[1,2]
  FP <- matriz_covid_arbol[2,1]; VN <- matriz_covid_arbol[2,2]

  data.frame(
    exactitud = (VP + VN) / sum(matriz_covid_arbol),
    sensibilidad = ifelse(VP + FN == 0, NA, VP / (VP + FN)),
    especificidad = ifelse(VN + FP == 0, NA, VN / (VN + FP))
  )
}


## Interpretación
La principal ventaja del árbol es que transforma el modelo en una secuencia visible de decisiones. Su desventaja es que un solo árbol puede cambiar bastante ante pequeñas variaciones de la muestra; esto motiva Random Forest.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=UperQjxBYEY) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-07/capitulo-07-arboles-presentacion.pdf) | [Descargar PDF](recursos/capitulo-07/capitulo-07-arboles-presentacion.pdf){download="capitulo-07-arboles-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-07/capitulo-07-arboles-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-07/capitulo-07-arboles-presentacion.pptx){download="capitulo-07-arboles-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-07/capitulo-07-arboles-infografia.png) | [Descargar PNG](recursos/capitulo-07/capitulo-07-arboles-infografia.png){download="capitulo-07-arboles-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/07-arboles-decision.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 7](recursos/capitulo-07/capitulo-07-arboles-infografia.png)](recursos/capitulo-07/capitulo-07-arboles-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/07-arboles-decision.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=UperQjxBYEY>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

Los árboles de decisión son modelos intuitivos, visuales e interpretables. Este capítulo prepara el camino para modelos basados en muchos árboles, como Random Forest.

## Referencias fundamentales de los árboles de decisión

La metodología CART fue desarrollada sistemáticamente por @breiman1984classification. El algoritmo ID3 y la inducción de árboles mediante ganancia de información fueron presentados por @quinlan1986induction. Para una introducción contemporánea con ejemplos en R puede consultarse @james2021islr.


# Random Forest para clasificación

La formulación matemática de **bootstrap, ensambles y Random Forest** se desarrolla con mayor profundidad
en los capítulos 7 y 14 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar Random Forest, entrenar un modelo en R, interpretar importancia de variables y evaluar desempeño.

## Introducción

Random Forest combina muchos árboles de decisión. Cada árbol aprende sobre una muestra aleatoria de los datos y la predicción final se obtiene por votación mayoritaria.

## Fundamento matemático

Si se construyen $B$ árboles, la predicción final es:

$$
\hat{y}(x) = moda\{\hat{y}^{(1)}(x), \hat{y}^{(2)}(x), \ldots, \hat{y}^{(B)}(x)\}
$$

## Cargar paquetes y datos


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
library(ranger)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Muestra de trabajo


In [ ]:
if (!is.null(atus_ml)) {
  set.seed(123)
  atus_rf_base <- atus_ml |>
    sample_n(min(30000, nrow(atus_ml))) |>
    mutate(
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños")),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI)
    ) |>
    na.omit()

  data.frame(filas = nrow(atus_rf_base), columnas = ncol(atus_rf_base))
}

if (exists("atus_rf_base")) {
  ggplot(atus_rf_base, aes(x = accidente_con_victimas, fill = accidente_con_victimas)) +
    geom_bar(width = 0.7) +
    escala_clases_fill() +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Distribución de la variable respuesta",
      subtitle = "Muestra para Random Forest",
      x = "Clase",
      y = "Número de accidentes"
    ) +
    tema_libro() +
    theme(legend.position = "none")
}


## Explicación del código
Se toma una muestra de trabajo para mantener el tiempo de ejecución razonable. El modelo Random Forest suele ser más pesado que un solo árbol.

## División en entrenamiento y prueba


In [ ]:
if (exists("atus_rf_base")) {
  set.seed(123)
  idx <- sample(1:nrow(atus_rf_base), size = round(0.7 * nrow(atus_rf_base)))
  entrenamiento <- atus_rf_base[idx, ]
  prueba <- atus_rf_base[-idx, ]

  data.frame(conjunto = c("Entrenamiento", "Prueba"), registros = c(nrow(entrenamiento), nrow(prueba)))
}


## Ajustar Random Forest


In [ ]:
if (exists("entrenamiento")) {
  set.seed(123)

  modelo_rf <- ranger(
    accidente_con_victimas ~ MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = entrenamiento,
    num.trees = 200,
    mtry = 3,
    min.node.size = 20,
    importance = "impurity",
    probability = TRUE,
    seed = 123
  )

  modelo_rf
}


## Interpretación del resultado
El resumen del modelo muestra cuántos árboles se construyeron, cuántas variables se probaron en cada división y el error fuera de bolsa.

## Importancia de variables


In [ ]:
if (exists("modelo_rf")) {
  importancia_rf <- data.frame(
    variable = names(modelo_rf$variable.importance),
    importancia = as.numeric(modelo_rf$variable.importance)
  ) |>
    arrange(desc(importancia))

  importancia_rf
}

if (exists("importancia_rf")) {
  ggplot(importancia_rf, aes(x = reorder(variable, importancia), y = importancia)) +
    geom_col(fill = col_azul, width = 0.75) +
    coord_flip() +
    labs(title = "Importancia de variables en Random Forest", x = "Variable", y = "Importancia") +
    tema_libro()
}


## Explicación del código
La importancia de variables muestra qué predictores aportan más a la reducción de impureza en los árboles del bosque.

## Predicción y métricas


In [ ]:
if (exists("modelo_rf")) {
  predicciones_rf <- predict(modelo_rf, data = prueba)
  probabilidades_rf <- predicciones_rf$predictions[, "Con víctimas"]
  clases_rf <- colnames(predicciones_rf$predictions)[max.col(predicciones_rf$predictions, ties.method = "first")]
  clases_rf <- factor(clases_rf, levels = c("Con víctimas", "Solo daños"))
  real_rf <- factor(prueba$accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))

  matriz_confusion_rf <- table(Real = real_rf, Predicho = clases_rf)
  matriz_confusion_rf
}

if (exists("matriz_confusion_rf")) {
  VP <- matriz_confusion_rf["Con víctimas", "Con víctimas"]
  FN <- matriz_confusion_rf["Con víctimas", "Solo daños"]
  FP <- matriz_confusion_rf["Solo daños", "Con víctimas"]
  VN <- matriz_confusion_rf["Solo daños", "Solo daños"]

  data.frame(exactitud=(VP+VN)/(VP+FN+FP+VN), sensibilidad=VP/(VP+FN), especificidad=VN/(VN+FP))
}


## Interpretación del resultado
La sensibilidad indica qué tan bien detecta accidentes con víctimas. La especificidad indica qué tan bien reconoce accidentes de solo daños.

## Distribución de probabilidades predichas


In [ ]:
if (exists("probabilidades_rf")) {
  datos_prob_rf <- data.frame(probabilidad = probabilidades_rf, clase_real = real_rf)

  ggplot(datos_prob_rf, aes(x = probabilidad, fill = clase_real)) +
    geom_histogram(bins = 30, alpha = 0.75, position = "identity") +
    escala_clases_fill(name = "Clase real") +
    labs(
      title = "Distribución de probabilidades predichas",
      subtitle = "Random Forest",
      x = "Probabilidad predicha de 'Con víctimas'",
      y = "Frecuencia"
    ) +
    tema_libro()
}


## Laboratorio interactivo: votación de un bosque

Este simulador muestra cómo múltiples árboles emiten votos y cómo la predicción
del bosque se estabiliza al aumentar el número de árboles.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El laboratorio permite cambiar el número de árboles y observar cómo se
estabiliza la proporción acumulada de votos del bosque.

## Caso aplicado B: Random Forest con COVID-19

Ahora aplicamos un bosque aleatorio a la misma muestra COVID utilizada en k-NN y árboles. Esto permite comparar un árbol individual con un ensamble de muchos árboles.

> **Uso académico:** las importancias y predicciones corresponden a una muestra educativa y no deben usarse para decisiones médicas.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_rf <- read_csv(ruta_covid, show_col_types = FALSE) |>
    select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
           OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_rf <- covid_rf |>
    sample_n(min(12000, nrow(covid_rf))) |>
    mutate(MURIO = factor(MURIO, levels = c(1, 0), labels = c("Defunción", "Sin defunción")))

  set.seed(2026)
  idx_covid <- sample(seq_len(nrow(covid_rf)), size = floor(0.80 * nrow(covid_rf)))
  covid_train <- covid_rf[idx_covid, ]
  covid_test <- covid_rf[-idx_covid, ]
}


In [ ]:
if (exists("covid_train")) {
  rf_covid <- ranger(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION + OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = covid_train,
    num.trees = 300,
    mtry = 3,
    min.node.size = 20,
    importance = "impurity",
    probability = TRUE,
    seed = 2026
  )
  rf_covid
}


In [ ]:
if (exists("rf_covid")) {
  imp_covid_rf <- data.frame(
    variable = names(rf_covid$variable.importance),
    importancia = as.numeric(rf_covid$variable.importance)
  ) |>
    arrange(desc(importancia))

  imp_covid_rf

  ggplot(imp_covid_rf, aes(x = reorder(variable, importancia), y = importancia)) +
    geom_col(fill = col_azul, width = 0.75) +
    coord_flip() +
    labs(title = "Importancia de variables: Random Forest COVID-19", x = "Variable", y = "Importancia") +
    tema_libro()
}


In [ ]:
if (exists("rf_covid")) {
  pred_prob_covid_rf <- predict(rf_covid, data = covid_test)$predictions
  pred_clase_covid_rf <- colnames(pred_prob_covid_rf)[max.col(pred_prob_covid_rf, ties.method = "first")]

  matriz_covid_rf <- table(
    Real = factor(covid_test$MURIO, levels = c("Defunción", "Sin defunción")),
    Predicho = factor(pred_clase_covid_rf, levels = c("Defunción", "Sin defunción"))
  )
  matriz_covid_rf

  VP <- matriz_covid_rf[1,1]; FN <- matriz_covid_rf[1,2]
  FP <- matriz_covid_rf[2,1]; VN <- matriz_covid_rf[2,2]

  data.frame(
    exactitud = (VP + VN) / sum(matriz_covid_rf),
    sensibilidad = ifelse(VP + FN == 0, NA, VP / (VP + FN)),
    especificidad = ifelse(VN + FP == 0, NA, VN / (VN + FP))
  )
}


## Interpretación
Random Forest suele ser más estable que un árbol individual porque combina muchas decisiones parcialmente distintas. La comparación de métricas con el árbol del capítulo anterior permite observar esa ganancia de estabilidad.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=KELSgYttifs) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-08/capitulo-08-random-forest-presentacion.pdf) | [Descargar PDF](recursos/capitulo-08/capitulo-08-random-forest-presentacion.pdf){download="capitulo-08-random-forest-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-08/capitulo-08-random-forest-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-08/capitulo-08-random-forest-presentacion.pptx){download="capitulo-08-random-forest-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-08/capitulo-08-random-forest-infografia.png) | [Descargar PNG](recursos/capitulo-08/capitulo-08-random-forest-infografia.png){download="capitulo-08-random-forest-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/08-random-forest.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 8](recursos/capitulo-08/capitulo-08-random-forest-infografia.png)](recursos/capitulo-08/capitulo-08-random-forest-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/08-random-forest.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=KELSgYttifs>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

Random Forest mejora la estabilidad y la capacidad predictiva de un solo árbol al combinar muchos árboles. Es uno de los métodos más útiles para clasificación aplicada.

## Referencias fundamentales de Random Forest

El algoritmo Random Forest fue formalizado por @breiman2001random como un ensamble de árboles construido mediante remuestreo y selección aleatoria de variables. Una explicación didáctica y comparativa puede consultarse también en @james2021islr.


# Evaluación, validación y comparación de modelos

La formulación matemática de **matrices de confusión, métricas, validación y selección de modelos** se desarrolla con mayor profundidad
en los capítulos 6 y 7 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de:

- explicar por qué la exactitud no siempre es suficiente;
- construir e interpretar una matriz de confusión;
- calcular exactitud, sensibilidad, especificidad, precisión y valor F1;
- distinguir entrenamiento, validación y prueba;
- aplicar validación cruzada estratificada;
- comparar modelos con criterios predictivos y prácticos;
- seleccionar un modelo sin depender de una sola métrica.

## Introducción

Entrenar un modelo es apenas la mitad del trabajo. Después debemos responder una pregunta más delicada: **¿qué tan bien funcionará con accidentes que nunca vio durante el aprendizaje?**

Un modelo puede memorizar los datos de entrenamiento y parecer excelente, pero fallar con observaciones nuevas. Por eso la evaluación debe realizarse con datos separados y métricas apropiadas para el objetivo del problema.

En este capítulo se utiliza nuevamente la base preparada a partir de ATUS [@inegi_atus_2024].

## Cargar los datos


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (!file.exists(ruta_atus_ml)) {
  stop(
    paste(
      "No se encontró datos/atus_ml_preparado.csv.",
      "Renderice primero el capítulo de preparación de datos."
    )
  )
}

atus_ml <- read_csv(
  ruta_atus_ml,
  show_col_types = FALSE
)


## Preparar una muestra reproducible


In [ ]:
set.seed(123)

atus_eval <- atus_ml |>
  sample_n(min(30000, nrow(atus_ml))) |>
  mutate(
    accidente_con_victimas = factor(
      accidente_con_victimas,
      levels = c("Solo daños", "Con víctimas")
    ),
    MES = factor(MES),
    ID_HORA = as.numeric(ID_HORA),
    DIASEMANA = factor(DIASEMANA),
    TIPACCID = factor(TIPACCID),
    CAUSAACCI = factor(CAUSAACCI)
  ) |>
  na.omit()

prop.table(table(atus_eval$accidente_con_victimas))


La tabla de proporciones permite identificar si una clase es mucho más frecuente que la otra. Cuando existe desbalance, una exactitud alta puede resultar engañosa.

## Entrenamiento, validación y prueba

Los datos pueden dividirse en tres partes:

- **Entrenamiento:** se utiliza para estimar los parámetros del modelo.
- **Validación:** ayuda a elegir hiperparámetros y comparar alternativas.
- **Prueba:** se reserva para estimar el desempeño final.

En ejercicios introductorios es frecuente utilizar entrenamiento y prueba, mientras que la validación se realiza mediante validación cruzada dentro del conjunto de entrenamiento.

## División estratificada

Una división estratificada conserva aproximadamente la proporción de cada clase.


In [ ]:
set.seed(123)

indices_entrenamiento <- unlist(
  lapply(
    split(seq_len(nrow(atus_eval)), atus_eval$accidente_con_victimas),
    function(indices) {
      sample(indices, size = floor(0.70 * length(indices)))
    }
  )
)

entrenamiento <- atus_eval[indices_entrenamiento, ]
prueba <- atus_eval[-indices_entrenamiento, ]

prop.table(table(entrenamiento$accidente_con_victimas))
prop.table(table(prueba$accidente_con_victimas))


Los índices se separan por clase y después se toma 70 % de cada grupo. Así se reduce el riesgo de que una clase quede sobrerrepresentada en entrenamiento o prueba.

## Matriz de confusión

Para una clase positiva denominada **Con víctimas**, la matriz contiene:

- **Verdadero positivo (VP):** el accidente tenía víctimas y el modelo lo detectó.
- **Falso negativo (FN):** tenía víctimas, pero el modelo lo clasificó como solo daños.
- **Falso positivo (FP):** era de solo daños, pero el modelo predijo víctimas.
- **Verdadero negativo (VN):** era de solo daños y se clasificó correctamente.

| Clase real | Predicción positiva | Predicción negativa |
|---|---:|---:|
| Positiva | VP | FN |
| Negativa | FP | VN |

Un falso negativo puede ser especialmente importante cuando el propósito consiste en identificar accidentes con víctimas.

## Métricas principales

### Exactitud

$$
\text{Exactitud}=\frac{VP+VN}{VP+FN+FP+VN}
$$

Mide la proporción total de predicciones correctas.

### Sensibilidad

$$
\text{Sensibilidad}=\frac{VP}{VP+FN}
$$

Mide qué proporción de los accidentes con víctimas fue detectada.

### Especificidad

$$
\text{Especificidad}=\frac{VN}{VN+FP}
$$

Mide qué proporción de los accidentes de solo daños fue reconocida correctamente.

### Precisión

$$
\text{Precisión}=\frac{VP}{VP+FP}
$$

Indica qué proporción de las predicciones positivas realmente tenía víctimas.

### Valor F1

$$
F_1=2\frac{\text{Precisión}\times\text{Sensibilidad}}
{\text{Precisión}+\text{Sensibilidad}}
$$

El valor F1 equilibra precisión y sensibilidad.

## Función para calcular métricas


In [ ]:
calcular_metricas <- function(real, predicho, positiva = "Con víctimas") {
  real <- factor(real)
  predicho <- factor(predicho, levels = levels(real))

  negativas <- setdiff(levels(real), positiva)

  if (length(negativas) != 1) {
    stop("La función requiere exactamente dos clases.")
  }

  negativa <- negativas[1]
  matriz <- table(Real = real, Predicho = predicho)

  VP <- matriz[positiva, positiva]
  FN <- matriz[positiva, negativa]
  FP <- matriz[negativa, positiva]
  VN <- matriz[negativa, negativa]

  division_segura <- function(numerador, denominador) {
    if (
      is.na(numerador) ||
      is.na(denominador) ||
      denominador == 0
    ) {
      return(NA_real_)
    }

    as.numeric(numerador / denominador)
  }

  exactitud <- division_segura(VP + VN, VP + FN + FP + VN)
  sensibilidad <- division_segura(VP, VP + FN)
  especificidad <- division_segura(VN, VN + FP)
  precision <- division_segura(VP, VP + FP)
  f1 <- division_segura(
    2 * precision * sensibilidad,
    precision + sensibilidad
  )

  list(
    matriz = matriz,
    metricas = data.frame(
      exactitud = exactitud,
      sensibilidad = sensibilidad,
      especificidad = especificidad,
      precision = precision,
      f1 = f1
    )
  )
}


## Modelo de referencia: regla mayoritaria

Antes de celebrar un modelo complejo, conviene compararlo con una regla muy simple: predecir siempre la clase más frecuente.


In [ ]:
clase_mayoritaria <- names(
  which.max(table(entrenamiento$accidente_con_victimas))
)

prediccion_referencia <- factor(
  rep(clase_mayoritaria, nrow(prueba)),
  levels = levels(entrenamiento$accidente_con_victimas)
)

resultado_referencia <- calcular_metricas(
  real = prueba$accidente_con_victimas,
  predicho = prediccion_referencia
)

resultado_referencia$matriz
resultado_referencia$metricas


El modelo de referencia predice únicamente la clase mayoritaria. Si nunca predice la clase positiva **Con víctimas**, la precisión no puede calcularse porque su denominador es cero. En ese caso R muestra `NA`, que significa **métrica no definida**, pero el libro continúa procesándose normalmente.

Si 90 % de los accidentes perteneciera a una sola clase, predecir siempre esa clase produciría 90 % de exactitud sin aprender ninguna relación útil. Por eso deben revisarse sensibilidad, especificidad, precisión y F1.

## Evaluar una regresión logística


In [ ]:
modelo_logistico <- glm(
  accidente_con_victimas ~
    MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
  data = entrenamiento,
  family = binomial
)

probabilidad_logistica <- predict(
  modelo_logistico,
  newdata = prueba,
  type = "response"
)

# glm() modela la probabilidad del segundo nivel del factor.
clase_positiva_modelada <- levels(
  entrenamiento$accidente_con_victimas
)[2]

prediccion_logistica <- ifelse(
  probabilidad_logistica >= 0.50,
  clase_positiva_modelada,
  levels(entrenamiento$accidente_con_victimas)[1]
)

prediccion_logistica <- factor(
  prediccion_logistica,
  levels = levels(entrenamiento$accidente_con_victimas)
)

resultado_logistico <- calcular_metricas(
  real = prueba$accidente_con_victimas,
  predicho = prediccion_logistica
)

resultado_logistico$matriz
resultado_logistico$metricas


En una regresión logística binaria, `glm()` calcula la probabilidad del segundo nivel del factor respuesta. El orden de los niveles debe verificarse antes de transformar probabilidades en clases.

## Comparación inicial


In [ ]:
comparacion_inicial <- bind_rows(
  cbind(
    modelo = "Regla mayoritaria",
    resultado_referencia$metricas
  ),
  cbind(
    modelo = "Regresión logística",
    resultado_logistico$metricas
  )
)

comparacion_inicial


In [ ]:
comparacion_larga <- comparacion_inicial |>
  tidyr::pivot_longer(
    cols = -modelo,
    names_to = "metrica",
    values_to = "valor"
  )

ggplot(
  comparacion_larga,
  aes(x = metrica, y = valor, fill = modelo)
) +
  geom_col(position = "dodge") +
  scale_y_continuous(
    limits = c(0, 1),
    labels = scales::label_percent(accuracy = 1)
  ) +
  labs(
    title = "Un modelo debe superar una referencia sencilla",
    x = "Métrica",
    y = "Valor",
    fill = "Modelo"
  ) +
  tema_libro()


**Fuente:** Elaboración propia con datos del INEGI, ATUS 2024.

## Umbral de clasificación

El umbral de 0.50 no es una ley universal. Reducirlo puede aumentar la sensibilidad, aunque también puede generar más falsos positivos.


In [ ]:
umbrales <- seq(0.10, 0.90, by = 0.05)

metricas_umbral <- lapply(umbrales, function(umbral) {
  prediccion <- ifelse(
    probabilidad_logistica >= umbral,
    clase_positiva_modelada,
    levels(entrenamiento$accidente_con_victimas)[1]
  )

  prediccion <- factor(
    prediccion,
    levels = levels(entrenamiento$accidente_con_victimas)
  )

  metricas <- calcular_metricas(
    prueba$accidente_con_victimas,
    prediccion
  )$metricas

  cbind(umbral = umbral, metricas)
}) |>
  bind_rows()

metricas_umbral


El mejor umbral depende del costo de cada error. Cuando omitir un accidente con víctimas es más grave que generar una alerta adicional, puede priorizarse la sensibilidad.

## Validación cruzada

La validación cruzada divide el conjunto de entrenamiento en varios pliegues. Cada pliegue funciona una vez como validación y los restantes como entrenamiento.

En validación cruzada de $K$ pliegues:

1. se divide la muestra en $K$ partes;
2. se entrena con $K-1$ partes;
3. se evalúa en la parte restante;
4. se repite hasta utilizar todos los pliegues;
5. se promedian las métricas.

### Crear pliegues estratificados


In [ ]:
crear_pliegues <- function(respuesta, k = 5, semilla = 123) {
  set.seed(semilla)

  pliegues <- integer(length(respuesta))

  for (clase in levels(factor(respuesta))) {
    indices <- which(respuesta == clase)
    etiquetas <- rep(seq_len(k), length.out = length(indices))
    pliegues[indices] <- sample(etiquetas)
  }

  pliegues
}

pliegue <- crear_pliegues(
  entrenamiento$accidente_con_victimas,
  k = 5
)

table(pliegue, entrenamiento$accidente_con_victimas)


### Validación cruzada de la regresión logística


In [ ]:
resultados_cv <- lapply(1:5, function(k) {
  datos_entrenamiento <- entrenamiento[pliegue != k, ]
  datos_validacion <- entrenamiento[pliegue == k, ]

  modelo <- glm(
    accidente_con_victimas ~
      MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = datos_entrenamiento,
    family = binomial
  )

  probabilidad <- predict(
    modelo,
    newdata = datos_validacion,
    type = "response"
  )

  clase_modelada <- levels(
    datos_entrenamiento$accidente_con_victimas
  )[2]

  prediccion <- ifelse(
    probabilidad >= 0.50,
    clase_modelada,
    levels(datos_entrenamiento$accidente_con_victimas)[1]
  )

  prediccion <- factor(
    prediccion,
    levels = levels(datos_entrenamiento$accidente_con_victimas)
  )

  metricas <- calcular_metricas(
    datos_validacion$accidente_con_victimas,
    prediccion
  )$metricas

  cbind(pliegue = k, metricas)
}) |>
  bind_rows()

resultados_cv


In [ ]:
resumen_cv <- resultados_cv |>
  summarise(
    across(
      where(is.numeric) & !matches("pliegue"),
      list(
        media = ~ mean(.x, na.rm = TRUE),
        desviacion = ~ sd(.x, na.rm = TRUE)
      )
    )
  )

resumen_cv


La media resume el desempeño esperado y la desviación estándar muestra su estabilidad. Dos modelos con medias parecidas pueden diferir mucho si uno presenta resultados más variables entre pliegues.

## Cómo comparar varios modelos

Para comparar regresión logística, k-NN, árbol de decisión y Random Forest conviene utilizar:

1. la misma muestra;
2. la misma variable respuesta;
3. la misma partición de entrenamiento y prueba;
4. el mismo criterio para definir la clase positiva;
5. las mismas métricas;
6. una validación cruzada equivalente;
7. tiempos de entrenamiento y predicción;
8. facilidad de interpretación.

Una tabla de decisión puede tener esta estructura:

| Modelo | Exactitud | Sensibilidad | Especificidad | F1 | Interpretación | Costo computacional |
|---|---:|---:|---:|---:|---|---|
| Regresión logística |  |  |  |  | Alta | Bajo |
| k-NN |  |  |  |  | Media-baja | Medio |
| Árbol de decisión |  |  |  |  | Alta | Bajo |
| Random Forest |  |  |  |  | Media | Alto |

## Selección del modelo final

No existe un modelo universalmente mejor. La selección depende de la finalidad:

- Si se requiere explicar el efecto de las variables, puede preferirse regresión logística.
- Si se necesita una regla visual y fácil de comunicar, un árbol puede ser apropiado.
- Si se prioriza capacidad predictiva y estabilidad, Random Forest puede resultar competitivo.
- Si el costo de los falsos negativos es alto, debe priorizarse sensibilidad.
- Si las alertas falsas son costosas, precisión y especificidad cobran mayor importancia.

El mejor modelo no es necesariamente el que tiene la mayor exactitud, sino el que responde mejor al objetivo, controla los errores relevantes y mantiene un desempeño estable con datos nuevos.

## Actividad guiada

1. Ejecute la regresión logística con umbrales de 0.30, 0.50 y 0.70.
2. Registre la matriz de confusión de cada caso.
3. Compare sensibilidad, especificidad, precisión y F1.
4. Explique qué umbral elegiría para detectar accidentes con víctimas.
5. Justifique su decisión considerando el costo de falsos negativos y falsos positivos.

## Actividades para el lector

1. Construya una función que calcule la exactitud balanceada:

   $$
   \text{Exactitud balanceada}=\frac{\text{Sensibilidad}+\text{Especificidad}}{2}
   $$

2. Compare la variabilidad de las métricas en validación cruzada.
3. Repita el procedimiento con un árbol de decisión.
4. Elabore una tabla comparativa con los cuatro modelos estudiados.
5. Escriba una recomendación técnica de no más de 200 palabras.

## Resumen del capítulo

En este capítulo aprendimos que evaluar un modelo requiere datos no utilizados durante el entrenamiento, una clase positiva bien definida y varias métricas. También construimos una regla de referencia, estudiamos el efecto del umbral y aplicamos validación cruzada estratificada. Estas herramientas permiten comparar modelos de forma más justa y elegirlos según el objetivo real del análisis.

## Referencias fundamentales sobre evaluación

La validación cruzada y su uso para estimar el desempeño de modelos fueron examinados comparativamente por @kohavi1995crossvalidation. Para el análisis mediante curvas ROC y AUC, una referencia ampliamente utilizada es @fawcett2006roc. Una visión integrada de partición de datos, remuestreo y comparación de modelos aparece en @james2021islr.

## Laboratorio interactivo: matriz de confusión y métricas

Modifica los cuatro valores de la matriz de confusión y observa cómo cambian
exactitud, sensibilidad, especificidad, precisión y F1.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

La calculadora permite editar la matriz de confusión y obtener automáticamente
exactitud, sensibilidad, especificidad, precisión y F1.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=ShiKrfhIfpY) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pdf) | [Descargar PDF](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pdf){download="capitulo-09-evaluacion-modelos-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pptx){download="capitulo-09-evaluacion-modelos-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png) | [Descargar PNG](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png){download="capitulo-09-evaluacion-modelos-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/09-evaluacion-modelos.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 9](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png)](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/09-evaluacion-modelos.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=ShiKrfhIfpY>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Caso aplicado B: evaluación del modelo COVID-19

La evaluación debe realizarse con observaciones que no participaron en el
ajuste del modelo.


In [ ]:
library(readr)
library(dplyr)
library(tidyr)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

covid_modelo <- covid |>
  select(
    MURIO,
    EDAD,
    NEUMONIA,
    DIABETES,
    HIPERTENSION,
    OBESIDAD,
    RENAL_CRONICA
  ) |>
  drop_na()

set.seed(2026)

indice_entrenamiento <- sample(
  seq_len(nrow(covid_modelo)),
  size = floor(0.80 * nrow(covid_modelo))
)

covid_train <- covid_modelo[indice_entrenamiento, ]
covid_test <- covid_modelo[-indice_entrenamiento, ]

modelo_covid <- glm(
  MURIO ~ EDAD + NEUMONIA + DIABETES +
    HIPERTENSION + OBESIDAD + RENAL_CRONICA,
  data = covid_train,
  family = binomial()
)

covid_test$probabilidad <- predict(
  modelo_covid,
  newdata = covid_test,
  type = "response"
)

umbral <- 0.50

covid_test$prediccion <- ifelse(
  covid_test$probabilidad >= umbral,
  1,
  0
)

VP <- sum(
  covid_test$prediccion == 1 &
    covid_test$MURIO == 1
)

VN <- sum(
  covid_test$prediccion == 0 &
    covid_test$MURIO == 0
)

FP <- sum(
  covid_test$prediccion == 1 &
    covid_test$MURIO == 0
)

FN <- sum(
  covid_test$prediccion == 0 &
    covid_test$MURIO == 1
)

matriz_covid <- matrix(
  c(VN, FP, FN, VP),
  nrow = 2,
  byrow = TRUE,
  dimnames = list(
    Real = c("Clase 0", "Clase 1"),
    Predicha = c("Clase 0", "Clase 1")
  )
)

matriz_covid


### Métricas


In [ ]:
exactitud <- (VP + VN) / (VP + VN + FP + FN)
sensibilidad <- VP / (VP + FN)
especificidad <- VN / (VN + FP)
precision <- VP / (VP + FP)
f1 <- 2 * precision * sensibilidad /
  (precision + sensibilidad)
balanced_accuracy <- (
  sensibilidad + especificidad
) / 2

data.frame(
  exactitud,
  sensibilidad,
  especificidad,
  precision,
  f1,
  balanced_accuracy
)


### Comparación de umbrales


In [ ]:
evaluar_umbral <- function(umbral) {
  pred <- ifelse(
    covid_test$probabilidad >= umbral,
    1,
    0
  )

  vp <- sum(pred == 1 & covid_test$MURIO == 1)
  vn <- sum(pred == 0 & covid_test$MURIO == 0)
  fp <- sum(pred == 1 & covid_test$MURIO == 0)
  fn <- sum(pred == 0 & covid_test$MURIO == 1)

  data.frame(
    umbral = umbral,
    sensibilidad = vp / (vp + fn),
    especificidad = vn / (vn + fp),
    precision = vp / (vp + fp)
  )
}

resultados_umbrales <- do.call(
  rbind,
  lapply(
    seq(0.10, 0.90, by = 0.05),
    evaluar_umbral
  )
)

resultados_umbrales


En un problema de detección de riesgo, disminuir el umbral suele aumentar la
sensibilidad, pero también puede generar más falsos positivos. El umbral no es
una constante universal: depende del objetivo y del costo de cada error.

## Comparación integrada de modelos con COVID-19

Los capítulos anteriores aplicaron distintos algoritmos a COVID-19. Para compararlos de manera más coherente construiremos ahora un **benchmark didáctico común**: todos los modelos usarán las mismas variables, la misma muestra, la misma partición de entrenamiento/prueba y las mismas métricas.

> **Uso académico:** esta comparación sirve para estudiar diferencias entre algoritmos. No es una evaluación clínica ni debe utilizarse para pronóstico individual.

### Una muestra balanceada para comparar algoritmos

La variable `MURIO` está muy desbalanceada en la base original. Si evaluáramos únicamente exactitud sobre una muestra aleatoria, un modelo podría obtener un valor alto simplemente prediciendo casi siempre la clase mayoritaria.

Para este ejercicio tomamos, de forma reproducible, hasta 2 500 observaciones de cada clase. Esto produce un banco de pruebas balanceado. **Las métricas obtenidas describen este benchmark educativo y no la prevalencia de defunción en la población.**


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_comp_base <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(
      MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
      OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES
    ) |>
    tidyr::drop_na() |>
    dplyr::mutate(
      MURIO = factor(
        MURIO,
        levels = c(0, 1),
        labels = c("Sin defunción", "Defunción")
      )
    )

  set.seed(2026)
  n_por_clase <- min(
    2500,
    min(table(covid_comp_base$MURIO))
  )

  covid_comp <- covid_comp_base |>
    dplyr::group_by(MURIO) |>
    dplyr::slice_sample(n = n_por_clase) |>
    dplyr::ungroup()

  set.seed(2026)
  idx_train <- unlist(lapply(
    split(seq_len(nrow(covid_comp)), covid_comp$MURIO),
    function(i) sample(i, floor(0.80 * length(i)))
  ))

  train_comp <- covid_comp[idx_train, ]
  test_comp <- covid_comp[-idx_train, ]

  prop.table(table(train_comp$MURIO))
  prop.table(table(test_comp$MURIO))
}


### Función común de métricas


In [ ]:
metricas_covid_comunes <- function(real, predicho) {
  niveles <- c("Sin defunción", "Defunción")
  real <- factor(real, levels = niveles)
  predicho <- factor(predicho, levels = niveles)
  m <- table(Real = real, Predicho = predicho)

  VP <- m["Defunción", "Defunción"]
  FN <- m["Defunción", "Sin defunción"]
  FP <- m["Sin defunción", "Defunción"]
  VN <- m["Sin defunción", "Sin defunción"]

  div <- function(a, b) {
    if (is.na(b) || b == 0) return(NA_real_)
    as.numeric(a / b)
  }

  exactitud <- div(VP + VN, sum(m))
  sensibilidad <- div(VP, VP + FN)
  especificidad <- div(VN, VN + FP)
  precision <- div(VP, VP + FP)
  f1 <- if (is.na(precision) || is.na(sensibilidad) ||
            precision + sensibilidad == 0) {
    NA_real_
  } else {
    2 * precision * sensibilidad / (precision + sensibilidad)
  }

  data.frame(
    exactitud = exactitud,
    sensibilidad = sensibilidad,
    especificidad = especificidad,
    precision = precision,
    f1 = f1
  )
}


### 1. Regresión logística


In [ ]:
if (exists("train_comp")) {
  modelo_log_comp <- glm(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    family = binomial
  )

  prob_log_comp <- predict(modelo_log_comp, test_comp, type = "response")
  pred_log_comp <- factor(
    ifelse(prob_log_comp >= 0.50, "Defunción", "Sin defunción"),
    levels = levels(train_comp$MURIO)
  )
}


### 2. k-NN


In [ ]:
if (exists("train_comp")) {
  vars_comp <- c(
    "EDAD", "NEUMONIA", "DIABETES", "HIPERTENSION",
    "OBESIDAD", "RENAL_CRONICA", "NUM_COMORBILIDADES"
  )

  X_train <- as.matrix(train_comp[vars_comp])
  X_test <- as.matrix(test_comp[vars_comp])

  medias <- apply(X_train, 2, mean)
  desv <- apply(X_train, 2, sd)
  desv[desv == 0] <- 1

  X_train_z <- scale(X_train, center = medias, scale = desv)
  X_test_z <- scale(X_test, center = medias, scale = desv)

  pred_knn_comp <- class::knn(
    train = X_train_z,
    test = X_test_z,
    cl = train_comp$MURIO,
    k = 11
  )
}


### 3. Árbol de decisión


In [ ]:
if (exists("train_comp")) {
  modelo_arbol_comp <- rpart::rpart(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    method = "class",
    control = rpart::rpart.control(
      cp = 0.002,
      minsplit = 60,
      minbucket = 20,
      maxdepth = 5
    )
  )

  pred_arbol_comp <- predict(
    modelo_arbol_comp,
    test_comp,
    type = "class"
  )
}


### 4. Random Forest


In [ ]:
if (exists("train_comp")) {
  set.seed(2026)
  modelo_rf_comp <- ranger::ranger(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    num.trees = 300,
    mtry = 3,
    min.node.size = 20,
    classification = TRUE,
    probability = FALSE,
    seed = 2026
  )

  pred_rf_comp <- predict(modelo_rf_comp, test_comp)$predictions
}


### 5. SVM radial


In [ ]:
if (exists("train_comp")) {
  set.seed(2026)
  modelo_svm_comp <- e1071::svm(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    kernel = "radial",
    cost = 1,
    gamma = 1 / 7,
    scale = TRUE
  )

  pred_svm_comp <- predict(modelo_svm_comp, test_comp)
}


### 6. Naive Bayes


In [ ]:
if (exists("train_comp")) {
  train_nb_comp <- train_comp |>
    dplyr::mutate(
      NEUMONIA = factor(NEUMONIA),
      DIABETES = factor(DIABETES),
      HIPERTENSION = factor(HIPERTENSION),
      OBESIDAD = factor(OBESIDAD),
      RENAL_CRONICA = factor(RENAL_CRONICA)
    )

  test_nb_comp <- test_comp |>
    dplyr::mutate(
      NEUMONIA = factor(NEUMONIA, levels = levels(train_nb_comp$NEUMONIA)),
      DIABETES = factor(DIABETES, levels = levels(train_nb_comp$DIABETES)),
      HIPERTENSION = factor(HIPERTENSION, levels = levels(train_nb_comp$HIPERTENSION)),
      OBESIDAD = factor(OBESIDAD, levels = levels(train_nb_comp$OBESIDAD)),
      RENAL_CRONICA = factor(RENAL_CRONICA, levels = levels(train_nb_comp$RENAL_CRONICA))
    )

  modelo_nb_comp <- e1071::naiveBayes(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_nb_comp,
    laplace = 1
  )

  pred_nb_comp <- predict(modelo_nb_comp, test_nb_comp, type = "class")
}


### 7. Red neuronal


In [ ]:
if (exists("train_comp")) {
  train_nn_comp <- as.data.frame(X_train_z)
  test_nn_comp <- as.data.frame(X_test_z)
  train_nn_comp$MURIO_NUM <- ifelse(train_comp$MURIO == "Defunción", 1, 0)

  set.seed(2026)
  modelo_nn_comp <- neuralnet::neuralnet(
    MURIO_NUM ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_nn_comp,
    hidden = 5,
    linear.output = FALSE,
    lifesign = "none",
    stepmax = 1e6
  )

  prob_nn_comp <- as.numeric(
    neuralnet::compute(
      modelo_nn_comp,
      test_nn_comp[vars_comp]
    )$net.result[, 1]
  )

  pred_nn_comp <- factor(
    ifelse(prob_nn_comp >= 0.50, "Defunción", "Sin defunción"),
    levels = levels(train_comp$MURIO)
  )
}


### Tabla comparativa final


In [ ]:
if (exists("pred_nn_comp")) {
  comparacion_covid_modelos <- dplyr::bind_rows(
    cbind(modelo = "Regresión logística", metricas_covid_comunes(test_comp$MURIO, pred_log_comp)),
    cbind(modelo = "k-NN (k=11)", metricas_covid_comunes(test_comp$MURIO, pred_knn_comp)),
    cbind(modelo = "Árbol de decisión", metricas_covid_comunes(test_comp$MURIO, pred_arbol_comp)),
    cbind(modelo = "Random Forest", metricas_covid_comunes(test_comp$MURIO, pred_rf_comp)),
    cbind(modelo = "SVM radial", metricas_covid_comunes(test_comp$MURIO, pred_svm_comp)),
    cbind(modelo = "Naive Bayes", metricas_covid_comunes(test_comp$MURIO, pred_nb_comp)),
    cbind(modelo = "Red neuronal", metricas_covid_comunes(test_comp$MURIO, pred_nn_comp))
  )

  comparacion_covid_modelos |>
    dplyr::mutate(
      dplyr::across(
        c(exactitud, sensibilidad, especificidad, precision, f1),
        ~ round(.x, 3)
      )
    )
}


In [ ]:
if (exists("comparacion_covid_modelos")) {
  comp_larga <- comparacion_covid_modelos |>
    tidyr::pivot_longer(
      cols = c(exactitud, sensibilidad, especificidad, precision, f1),
      names_to = "metrica",
      values_to = "valor"
    )

  ggplot2::ggplot(
    comp_larga,
    ggplot2::aes(x = modelo, y = valor, fill = metrica)
  ) +
    ggplot2::geom_col(position = "dodge") +
    ggplot2::coord_flip() +
    ggplot2::scale_y_continuous(limits = c(0, 1)) +
    ggplot2::labs(
      title = "Comparación común de modelos con COVID-19",
      subtitle = "Misma muestra balanceada, mismas variables y misma partición 80/20",
      x = NULL,
      y = "Valor de la métrica",
      fill = "Métrica"
    ) +
    tema_libro()
}


El modelo con mayor exactitud no necesariamente es el mejor para todos los objetivos. La sensibilidad prioriza detectar defunciones; la especificidad prioriza reconocer correctamente los casos sin defunción; precisión y F1 ofrecen otras perspectivas. Además, aquí se usaron hiperparámetros didácticos fijos, no una búsqueda exhaustiva de optimización.

### Lectura recomendada de la tabla

Al comparar los modelos conviene preguntar:

1. ¿Cuál ofrece el mejor equilibrio entre sensibilidad y especificidad?
2. ¿Algún modelo gana en exactitud pero pierde mucha sensibilidad?
3. ¿La mayor complejidad de Random Forest, SVM o la red neuronal produce una mejora suficiente frente a la regresión logística?
4. ¿Un modelo sencillo sería preferible si ofrece resultados parecidos y mayor interpretabilidad?
5. ¿Cambiaría la conclusión si la muestra conservara la fuerte desproporción original entre clases?

Esta comparación cierra la ruta supervisada del caso COVID-19 y conecta los capítulos de algoritmos con una evaluación común y reproducible.


# Máquinas de Vectores de Soporte (SVM)

La formulación matemática de **margen, optimización convexa, condiciones KKT y kernels** se desarrolla con mayor profundidad
en los capítulos 4 y 15 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de:

- explicar la idea del hiperplano de separación y del margen máximo;
- identificar los vectores de soporte;
- distinguir entre una SVM lineal y una SVM con kernel;
- comprender el papel de los parámetros `cost`, `gamma` y del tipo de kernel;
- entrenar modelos SVM en R;
- evaluar una SVM mediante una matriz de confusión y métricas de clasificación;
- utilizar ponderación de clases cuando la variable respuesta está desbalanceada;
- comparar una SVM lineal con una SVM de kernel radial.

## Introducción

Las **máquinas de vectores de soporte**, conocidas como **SVM** por *Support Vector Machines*, son métodos supervisados utilizados principalmente para clasificación. Su propósito es construir una frontera que separe las clases con el margen más amplio posible.

La idea es sencilla de visualizar: si dos grupos pueden separarse mediante una línea, existen muchas líneas posibles, pero la SVM busca aquella que deja la mayor distancia entre la frontera y las observaciones más cercanas de cada clase.

Esas observaciones cercanas reciben el nombre de **vectores de soporte**. Aunque el conjunto de datos contenga miles de registros, los vectores de soporte son los que determinan directamente la posición de la frontera.

En este capítulo se emplean primero datos simulados para entender la geometría del método y después la base preparada de ATUS [@inegi_atus_2024].

## Intuición geométrica

Supongamos que cada observación tiene dos variables, $x_1$ y $x_2$. Una frontera lineal puede escribirse como:

$$
w_1x_1+w_2x_2+b=0
$$

donde:

- $w_1$ y $w_2$ determinan la orientación de la frontera;
- $b$ desplaza la frontera;
- el signo de la expresión determina de qué lado queda cada observación.

La SVM busca una frontera con **margen máximo**. En una separación ideal se desea que:

$$
y_i(\mathbf{w}^{T}\mathbf{x}_i+b)\geq 1
$$

para todas las observaciones, donde $y_i$ representa la clase codificada como $-1$ o $1$.

El ancho del margen es proporcional a:

$$
\frac{2}{\lVert\mathbf{w}\rVert}
$$

Por ello, maximizar el margen equivale a minimizar la magnitud de $\mathbf{w}$.

Una SVM no busca solamente clasificar bien los datos de entrenamiento. Busca una frontera que conserve la mayor separación posible entre las clases, con la intención de generalizar mejor ante observaciones nuevas.

## Crear un ejemplo didáctico


In [ ]:
set.seed(123)

n <- 120

datos_svm <- data.frame(
  x1 = c(rnorm(n / 2, mean = -1.5, sd = 0.8),
         rnorm(n / 2, mean =  1.5, sd = 0.8)),
  x2 = c(rnorm(n / 2, mean = -1.0, sd = 0.9),
         rnorm(n / 2, mean =  1.0, sd = 0.9)),
  clase = factor(rep(c("Clase A", "Clase B"), each = n / 2))
)

head(datos_svm)


Se generan dos grupos artificiales. El primero se concentra alrededor de valores negativos y el segundo alrededor de valores positivos. El ejemplo permite observar con claridad la frontera de clasificación.

## Visualizar las clases


In [ ]:
library(ggplot2)
source("util_graficas.R")

ggplot(datos_svm, aes(x = x1, y = x2, color = clase)) +
  geom_point(size = 2.5, alpha = 0.8) +
  labs(
    title = "Datos simulados para clasificación",
    subtitle = "Cada punto representa una observación",
    x = "Variable x1",
    y = "Variable x2",
    color = "Clase"
  ) +
  tema_libro()


## Instalar y cargar el paquete `e1071`


In [ ]:
if (!requireNamespace("e1071", quietly = TRUE)) {
  install.packages("e1071", repos = "https://cloud.r-project.org")
}

library(e1071)


El paquete `e1071` proporciona la función `svm()`, que permite entrenar modelos lineales y no lineales.

## Entrenar una SVM lineal


In [ ]:
modelo_svm_lineal <- svm(
  clase ~ x1 + x2,
  data = datos_svm,
  kernel = "linear",
  cost = 1,
  scale = TRUE
)

modelo_svm_lineal


Los argumentos principales son:

- `kernel = "linear"`: utiliza una frontera lineal;
- `cost = 1`: establece la penalización por errores;
- `scale = TRUE`: estandariza las variables numéricas antes del ajuste.

## Identificar los vectores de soporte


In [ ]:
indices_soporte <- modelo_svm_lineal$index
vectores_soporte <- datos_svm[indices_soporte, ]

nrow(vectores_soporte)
head(vectores_soporte)


Los vectores de soporte son las observaciones más influyentes para establecer la frontera. Los puntos alejados del margen suelen tener poca o ninguna influencia directa sobre su posición.

## Dibujar la frontera de decisión


In [ ]:
rejilla <- expand.grid(
  x1 = seq(min(datos_svm$x1) - 0.5,
           max(datos_svm$x1) + 0.5,
           length.out = 180),
  x2 = seq(min(datos_svm$x2) - 0.5,
           max(datos_svm$x2) + 0.5,
           length.out = 180)
)

rejilla$prediccion <- predict(modelo_svm_lineal, newdata = rejilla)

ggplot() +
  geom_tile(
    data = rejilla,
    aes(x = x1, y = x2, fill = prediccion),
    alpha = 0.18
  ) +
  geom_point(
    data = datos_svm,
    aes(x = x1, y = x2, color = clase),
    size = 2.2
  ) +
  geom_point(
    data = vectores_soporte,
    aes(x = x1, y = x2),
    shape = 21,
    size = 4,
    stroke = 1.1,
    fill = NA
  ) +
  labs(
    title = "Frontera de decisión de una SVM lineal",
    subtitle = "Los círculos grandes identifican los vectores de soporte",
    x = "Variable x1",
    y = "Variable x2",
    fill = "Predicción",
    color = "Clase real"
  ) +
  tema_libro()


## El parámetro de costo

El parámetro `cost` controla la penalización asignada a las observaciones clasificadas incorrectamente.

- Un costo pequeño permite más errores y favorece un margen amplio.
- Un costo grande penaliza fuertemente los errores y puede producir una frontera más ajustada a los datos de entrenamiento.

No existe un valor universalmente mejor. Debe seleccionarse con validación cruzada.


In [ ]:
modelos_cost <- lapply(
  c(0.1, 1, 10),
  function(valor_cost) {
    svm(
      clase ~ x1 + x2,
      data = datos_svm,
      kernel = "linear",
      cost = valor_cost,
      scale = TRUE
    )
  }
)

resumen_cost <- data.frame(
  cost = c(0.1, 1, 10),
  vectores_soporte = vapply(
    modelos_cost,
    function(modelo) modelo$tot.nSV,
    numeric(1)
  )
)

resumen_cost


## Cuando una línea no es suficiente

Algunos conjuntos de datos no pueden separarse adecuadamente mediante una línea o un hiperplano. En esos casos, una SVM puede utilizar una función denominada **kernel**.

El kernel calcula similitudes entre observaciones y permite representar fronteras no lineales sin construir explícitamente todas las nuevas variables.

Los kernels más utilizados son:

| Kernel | Uso general |
|---|---|
| Lineal | Cuando la separación es aproximadamente lineal o existen muchas variables |
| Polinomial | Cuando se esperan relaciones de tipo polinómico |
| Radial | Cuando la frontera puede tener formas curvas y complejas |
| Sigmoide | Menos frecuente; guarda relación con funciones de activación |

## Kernel radial

El kernel radial utiliza una función semejante a:

$$
K(\mathbf{x}_i,\mathbf{x}_j)=
\exp\left(-\gamma\lVert\mathbf{x}_i-\mathbf{x}_j\rVert^2\right)
$$

El parámetro $\gamma$ controla el alcance de la influencia de cada observación:

- valores pequeños producen fronteras más suaves;
- valores grandes permiten fronteras más locales y complejas.

Una combinación de `cost` y `gamma` demasiado grande puede ajustar muy bien el entrenamiento, pero funcionar peor con datos nuevos. La validación cruzada ayuda a evitar esa elección.

## Aplicación con los datos ATUS

## Cargar la base preparada


In [ ]:
library(readr)
library(dplyr)

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (!file.exists(ruta_atus_ml)) {
  stop(
    paste(
      "No se encontró datos/atus_ml_preparado.csv.",
      "Renderice primero el capítulo de preparación de datos."
    )
  )
}

atus_ml <- read_csv(
  ruta_atus_ml,
  show_col_types = FALSE
)


## Preparar una muestra reproducible

Las SVM pueden requerir bastante memoria y tiempo cuando el conjunto es muy grande. Para fines didácticos se utiliza una muestra de hasta 6 000 accidentes.


In [ ]:
set.seed(123)

atus_svm <- atus_ml |>
  sample_n(min(6000, nrow(atus_ml))) |>
  transmute(
    accidente_con_victimas = factor(
      accidente_con_victimas,
      levels = c("Solo daños", "Con víctimas")
    ),
    MES = factor(MES),
    ID_HORA = as.numeric(ID_HORA),
    DIASEMANA = factor(DIASEMANA),
    TIPACCID = factor(TIPACCID),
    CAUSAACCI = factor(CAUSAACCI)
  ) |>
  na.omit()

prop.table(table(atus_svm$accidente_con_victimas))


La muestra reduce el tiempo de ejecución y hace posible repetir el ejercicio en computadoras personales y en Google Colab. Para un estudio definitivo se recomienda evaluar tamaños mayores y documentar los recursos computacionales utilizados.

## Dividir los datos de forma estratificada


In [ ]:
set.seed(123)

indices_entrenamiento <- unlist(
  lapply(
    split(seq_len(nrow(atus_svm)), atus_svm$accidente_con_victimas),
    function(indices) {
      sample(indices, size = floor(0.70 * length(indices)))
    }
  )
)

entrenamiento_svm <- atus_svm[indices_entrenamiento, ]
prueba_svm <- atus_svm[-indices_entrenamiento, ]

prop.table(table(entrenamiento_svm$accidente_con_victimas))
prop.table(table(prueba_svm$accidente_con_victimas))


## Calcular pesos para las clases

Si una clase aparece con menor frecuencia, la SVM puede recibir pesos inversamente proporcionales a sus frecuencias.


In [ ]:
frecuencias <- table(entrenamiento_svm$accidente_con_victimas)

pesos_clase <- sum(frecuencias) /
  (length(frecuencias) * frecuencias)

pesos_clase <- as.numeric(pesos_clase)
names(pesos_clase) <- names(frecuencias)

pesos_clase


La clase menos frecuente recibe un peso mayor, de modo que sus errores tengan más influencia durante el entrenamiento.

## Ajustar una SVM lineal


In [ ]:
set.seed(123)

modelo_svm_atus_lineal <- svm(
  accidente_con_victimas ~
    MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
  data = entrenamiento_svm,
  kernel = "linear",
  cost = 1,
  class.weights = pesos_clase,
  scale = TRUE
)

modelo_svm_atus_lineal


## Realizar predicciones


In [ ]:
prediccion_svm_lineal <- predict(
  modelo_svm_atus_lineal,
  newdata = prueba_svm
)

matriz_svm_lineal <- table(
  Real = prueba_svm$accidente_con_victimas,
  Predicho = prediccion_svm_lineal
)

matriz_svm_lineal


## Función segura para calcular métricas


In [ ]:
metricas_clasificacion <- function(
  real,
  predicho,
  positiva = "Con víctimas"
) {
  real <- factor(real)
  predicho <- factor(predicho, levels = levels(real))

  negativa <- setdiff(levels(real), positiva)

  if (length(negativa) != 1) {
    stop("La función requiere exactamente dos clases.")
  }

  negativa <- negativa[1]
  matriz <- table(Real = real, Predicho = predicho)

  VP <- matriz[positiva, positiva]
  FN <- matriz[positiva, negativa]
  FP <- matriz[negativa, positiva]
  VN <- matriz[negativa, negativa]

  dividir <- function(a, b) {
    if (is.na(a) || is.na(b) || b == 0) {
      return(NA_real_)
    }
    as.numeric(a / b)
  }

  exactitud <- dividir(VP + VN, VP + FN + FP + VN)
  sensibilidad <- dividir(VP, VP + FN)
  especificidad <- dividir(VN, VN + FP)
  precision <- dividir(VP, VP + FP)
  f1 <- dividir(
    2 * precision * sensibilidad,
    precision + sensibilidad
  )

  data.frame(
    exactitud = exactitud,
    sensibilidad = sensibilidad,
    especificidad = especificidad,
    precision = precision,
    f1 = f1
  )
}


## Evaluar la SVM lineal


In [ ]:
metricas_svm_lineal <- metricas_clasificacion(
  real = prueba_svm$accidente_con_victimas,
  predicho = prediccion_svm_lineal
)

metricas_svm_lineal


La sensibilidad indica la proporción de accidentes con víctimas detectada por el modelo. La especificidad indica la proporción de accidentes de solo daños reconocida correctamente. Cuando las clases están desbalanceadas, estas métricas suelen ser más informativas que la exactitud aislada.

## Ajustar una SVM con kernel radial


In [ ]:
set.seed(123)

modelo_svm_atus_radial <- svm(
  accidente_con_victimas ~
    MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
  data = entrenamiento_svm,
  kernel = "radial",
  cost = 1,
  gamma = 0.03,
  class.weights = pesos_clase,
  scale = TRUE
)

modelo_svm_atus_radial


## Evaluar la SVM radial


In [ ]:
prediccion_svm_radial <- predict(
  modelo_svm_atus_radial,
  newdata = prueba_svm
)

matriz_svm_radial <- table(
  Real = prueba_svm$accidente_con_victimas,
  Predicho = prediccion_svm_radial
)

matriz_svm_radial


In [ ]:
metricas_svm_radial <- metricas_clasificacion(
  real = prueba_svm$accidente_con_victimas,
  predicho = prediccion_svm_radial
)

metricas_svm_radial


## Comparar los dos modelos


In [ ]:
comparacion_svm <- bind_rows(
  transform(metricas_svm_lineal, modelo = "SVM lineal"),
  transform(metricas_svm_radial, modelo = "SVM radial")
) |>
  select(modelo, everything())

comparacion_svm


In [ ]:
comparacion_larga <- comparacion_svm |>
  tidyr::pivot_longer(
    cols = -modelo,
    names_to = "metrica",
    values_to = "valor"
  )

ggplot(
  comparacion_larga,
  aes(x = metrica, y = valor, fill = modelo)
) +
  geom_col(position = "dodge") +
  coord_cartesian(ylim = c(0, 1)) +
  labs(
    title = "Comparación de modelos SVM",
    subtitle = "Resultados sobre el conjunto de prueba",
    x = "Métrica",
    y = "Valor",
    fill = "Modelo"
  ) +
  tema_libro()


## Selección de hiperparámetros con validación cruzada

La función `tune.svm()` permite probar diferentes combinaciones de hiperparámetros. Para mantener un tiempo razonable se usa una cuadrícula pequeña.


In [ ]:
set.seed(123)

muestra_ajuste <- entrenamiento_svm |>
  sample_n(min(3000, nrow(entrenamiento_svm)))

ajuste_svm <- tune.svm(
  accidente_con_victimas ~
    MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
  data = muestra_ajuste,
  kernel = "radial",
  cost = c(0.5, 1, 2),
  gamma = c(0.01, 0.03, 0.05),
  tunecontrol = tune.control(cross = 5),
  scale = TRUE
)

ajuste_svm$best.parameters


In [ ]:
mejor_modelo_svm <- ajuste_svm$best.model

prediccion_mejor_svm <- predict(
  mejor_modelo_svm,
  newdata = prueba_svm
)

metricas_mejor_svm <- metricas_clasificacion(
  real = prueba_svm$accidente_con_victimas,
  predicho = prediccion_mejor_svm
)

metricas_mejor_svm


Los hiperparámetros deben seleccionarse usando exclusivamente los datos de entrenamiento. El conjunto de prueba se reserva para la evaluación final.

## Ventajas y limitaciones

### Ventajas

- puede construir fronteras lineales y no lineales;
- funciona bien en espacios con muchas variables;
- el margen máximo favorece la generalización;
- utiliza principalmente los vectores de soporte para definir la frontera;
- permite ponderar clases desbalanceadas.

### Limitaciones

- puede ser lenta con conjuntos de datos muy grandes;
- requiere elegir kernel e hiperparámetros;
- sus resultados son menos interpretables que los de una regresión logística o un árbol pequeño;
- la estandarización de variables numéricas es importante;
- una búsqueda extensa de hiperparámetros puede consumir muchos recursos.

## Recomendaciones prácticas

1. Comience con una SVM lineal como referencia.
2. Estandarice las variables numéricas.
3. Use validación cruzada para seleccionar `cost` y `gamma`.
4. Revise sensibilidad, especificidad, precisión y F1, no solo exactitud.
5. Utilice pesos de clase cuando exista desbalance.
6. Trabaje inicialmente con una muestra si el conjunto es muy grande.
7. Evalúe el modelo final una sola vez sobre datos de prueba no utilizados durante el ajuste.

## Actividad guiada

Modifique el código para comparar:

- `cost = 0.1`, `1` y `10` en la SVM lineal;
- `gamma = 0.01`, `0.05` y `0.10` en la SVM radial;
- modelos con y sin `class.weights`;
- muestras de 3 000 y 6 000 accidentes.

Registre para cada modelo:

- número de vectores de soporte;
- tiempo de entrenamiento;
- exactitud;
- sensibilidad;
- especificidad;
- precisión;
- valor F1.

## Ejercicios

1. Explique con sus propias palabras qué es un vector de soporte.
2. ¿Qué diferencia existe entre una frontera lineal y una frontera construida con kernel radial?
3. ¿Qué ocurre generalmente cuando `cost` es demasiado grande?
4. ¿Por qué es importante estandarizar variables numéricas?
5. Entrene una SVM lineal sin pesos de clase y compare su sensibilidad con el modelo ponderado.
6. Amplíe la cuadrícula de validación cruzada y explique si el mejor modelo cambia.
7. Compare la mejor SVM con la regresión logística y Random Forest estudiados anteriormente.
8. Analice si una mejora pequeña en exactitud justifica una pérdida importante de interpretabilidad.

## Conclusiones

Las máquinas de vectores de soporte buscan una frontera con margen amplio y se apoyan en un subconjunto de observaciones denominado vectores de soporte. Mediante kernels pueden representar relaciones no lineales y producir modelos predictivos potentes.

Sin embargo, una SVM requiere seleccionar cuidadosamente sus hiperparámetros y evaluar su desempeño con datos no utilizados durante el entrenamiento. En problemas con clases desbalanceadas, la ponderación de clases y el análisis conjunto de sensibilidad, especificidad, precisión y F1 son fundamentales.

**Fuente de los datos de aplicación:** elaboración propia con datos del INEGI, Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), 2024.

## Referencias fundamentales de SVM

Las máquinas de vectores de soporte fueron presentadas formalmente por @cortes1995support. El tutorial de @burges1998tutorial desarrolla la intuición geométrica, los márgenes y el uso de kernels. Una introducción aplicada en R puede consultarse en @james2021islr.

## Laboratorio interactivo: margen e hiperplano

Explora una frontera lineal de la forma
\(w_1x_1+w_2x_2+b=0\), sus márgenes y la clasificación de un punto nuevo.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

La versión web permite cambiar los pesos, el sesgo y un punto nuevo para
observar el hiperplano, los márgenes y la clase predicha.

## Caso aplicado B: SVM con COVID-19

En esta segunda ruta aplicada usamos datos abiertos de COVID-19 México 2022. El objetivo educativo es clasificar la variable `MURIO`, donde **1 representa defunción registrada y 0 ausencia de defunción registrada**, a partir de edad, neumonía, diabetes, hipertensión, obesidad, enfermedad renal crónica y número de comorbilidades.

> **Uso académico:** este ejercicio ilustra el funcionamiento de SVM. No debe interpretarse como herramienta clínica, pronóstico individual ni diagnóstico.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_svm <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
                  OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na() |>
    dplyr::mutate(
      MURIO = factor(MURIO, levels = c(0, 1),
                     labels = c("Sin defunción", "Defunción"))
    )

  set.seed(2026)
  covid_svm <- covid_svm |>
    dplyr::sample_n(min(8000, nrow(covid_svm)))

  set.seed(2026)
  idx_svm_covid <- unlist(lapply(
    split(seq_len(nrow(covid_svm)), covid_svm$MURIO),
    function(i) sample(i, floor(0.80 * length(i)))
  ))

  train_svm_covid <- covid_svm[idx_svm_covid, ]
  test_svm_covid <- covid_svm[-idx_svm_covid, ]
}


La división es estratificada para conservar aproximadamente la proporción de ambas clases. Además, `scale = TRUE` estandariza los predictores numéricos dentro de `svm()`.


In [ ]:
if (exists("train_svm_covid")) {
  frec_covid <- table(train_svm_covid$MURIO)
  pesos_covid <- sum(frec_covid) / (length(frec_covid) * frec_covid)

  modelo_svm_covid_lineal <- e1071::svm(
    MURIO ~ ., data = train_svm_covid,
    kernel = "linear", cost = 1,
    class.weights = pesos_covid,
    scale = TRUE
  )

  modelo_svm_covid_radial <- e1071::svm(
    MURIO ~ ., data = train_svm_covid,
    kernel = "radial", cost = 1,
    gamma = 1 / (ncol(train_svm_covid) - 1),
    class.weights = pesos_covid,
    scale = TRUE
  )

  pred_lin <- predict(modelo_svm_covid_lineal, test_svm_covid)
  pred_rad <- predict(modelo_svm_covid_radial, test_svm_covid)
}


In [ ]:
metricas_covid_binarias <- function(real, predicho, positiva = "Defunción") {
  real <- factor(real, levels = c("Sin defunción", "Defunción"))
  predicho <- factor(predicho, levels = levels(real))
  m <- table(Real = real, Predicho = predicho)
  VP <- m[positiva, positiva]
  FN <- m[positiva, "Sin defunción"]
  FP <- m["Sin defunción", positiva]
  VN <- m["Sin defunción", "Sin defunción"]
  div <- function(a,b) ifelse(b == 0, NA_real_, as.numeric(a/b))
  data.frame(
    exactitud = div(VP+VN, sum(m)),
    sensibilidad = div(VP, VP+FN),
    especificidad = div(VN, VN+FP)
  )
}

if (exists("pred_lin")) {
  comparacion_svm_covid <- dplyr::bind_rows(
    cbind(modelo = "SVM lineal", metricas_covid_binarias(test_svm_covid$MURIO, pred_lin)),
    cbind(modelo = "SVM radial", metricas_covid_binarias(test_svm_covid$MURIO, pred_rad))
  )
  comparacion_svm_covid
}


La comparación permite estudiar si una frontera no lineal aporta una ventaja real sobre la SVM lineal. En una respuesta desbalanceada, sensibilidad y especificidad deben revisarse junto con la exactitud.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=upiQi4xIBrQ) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-10/capitulo-10-svm-presentacion.pdf) | [Descargar PDF](recursos/capitulo-10/capitulo-10-svm-presentacion.pdf){download="capitulo-10-svm-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-10/capitulo-10-svm-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-10/capitulo-10-svm-presentacion.pptx){download="capitulo-10-svm-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-10/capitulo-10-svm-infografia.png) | [Descargar PNG](recursos/capitulo-10/capitulo-10-svm-infografia.png){download="capitulo-10-svm-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/10-svm.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 10](recursos/capitulo-10/capitulo-10-svm-infografia.png)](recursos/capitulo-10/capitulo-10-svm-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/10-svm.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=upiQi4xIBrQ>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)


# Clasificador Naive Bayes

La formulación matemática de **probabilidad condicional, teorema de Bayes y clasificación probabilística** se desarrolla con mayor profundidad
en los capítulos 3 y 6 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de:

- explicar el teorema de Bayes aplicado a clasificación;
- interpretar probabilidades previas, verosimilitudes y probabilidades posteriores;
- comprender el supuesto de independencia condicional;
- distinguir entre Naive Bayes categórico y gaussiano;
- construir un clasificador sencillo sin paquetes especializados;
- entrenar y evaluar un modelo Naive Bayes en R;
- aplicar el algoritmo a una muestra de la base ATUS;
- reconocer sus ventajas, limitaciones y condiciones de uso.

## Introducción

**Naive Bayes** es una familia de clasificadores probabilísticos basada en el teorema de Bayes. Su nombre incluye la palabra *naive* —ingenuo— porque supone que las variables predictoras son condicionalmente independientes una vez conocida la clase.

Este supuesto rara vez se cumple de manera exacta en datos reales. Sin embargo, el clasificador puede funcionar sorprendentemente bien incluso cuando existe cierta dependencia entre las variables [@domingos1997optimality; @hand2001idiots].

La principal fortaleza del método es que transforma la clasificación en una comparación de probabilidades. Para una observación nueva, se calcula qué clase resulta más probable dados los valores de sus predictores.

## Recordatorio del teorema de Bayes

Para una clase $C$ y un conjunto de características $X$, el teorema de Bayes establece:

$$
P(C\mid X)=\frac{P(X\mid C)P(C)}{P(X)}
$$

donde:

- $P(C)$ es la **probabilidad previa** de la clase;
- $P(X\mid C)$ es la **verosimilitud** de observar las características cuando la clase es $C$;
- $P(C\mid X)$ es la **probabilidad posterior** de la clase después de observar $X$;
- $P(X)$ es una constante común al comparar las clases.

Para clasificar no es necesario calcular explícitamente $P(X)$. Basta comparar:

$$
P(C\mid X)\propto P(X\mid C)P(C)
$$

## El supuesto ingenuo de independencia

Si una observación tiene predictores $X_1,X_2,\ldots,X_p$, Naive Bayes supone que, conocida la clase, sus contribuciones pueden multiplicarse:

$$
P(X_1,\ldots,X_p\mid C)
=\prod_{j=1}^{p}P(X_j\mid C)
$$

Por tanto:

$$
P(C\mid X_1,\ldots,X_p)
\propto P(C)\prod_{j=1}^{p}P(X_j\mid C)
$$

Naive Bayes calcula un puntaje probabilístico para cada clase. La observación se asigna a la clase cuyo producto entre probabilidad previa y verosimilitudes es mayor.

## Ejemplo manual con variables categóricas

Consideremos un pequeño conjunto de accidentes descritos por dos variables: condición climática y horario.


In [ ]:
datos_nb <- data.frame(
  clima = c("Seco", "Seco", "Lluvia", "Lluvia", "Seco",
            "Lluvia", "Seco", "Lluvia", "Seco", "Lluvia"),
  horario = c("Día", "Noche", "Día", "Noche", "Día",
              "Noche", "Noche", "Día", "Día", "Noche"),
  victimas = factor(
    c("No", "No", "Sí", "Sí", "No", "Sí", "Sí", "No", "No", "Sí")
  )
)

datos_nb


## Calcular las probabilidades previas


In [ ]:
previas <- prop.table(table(datos_nb$victimas))
previas


Las probabilidades previas representan la proporción inicial de cada clase antes de observar las variables predictoras.

## Calcular probabilidades condicionales


In [ ]:
prob_clima <- prop.table(
  table(datos_nb$clima, datos_nb$victimas),
  margin = 2
)

prob_horario <- prop.table(
  table(datos_nb$horario, datos_nb$victimas),
  margin = 2
)

prob_clima
prob_horario


## Clasificar una observación manualmente

Supongamos un accidente ocurrido con lluvia durante la noche. Comparamos los puntajes para las dos clases.


In [ ]:
puntaje_si <-
  previas["Sí"] *
  prob_clima["Lluvia", "Sí"] *
  prob_horario["Noche", "Sí"]

puntaje_no <-
  previas["No"] *
  prob_clima["Lluvia", "No"] *
  prob_horario["Noche", "No"]

c(Sí = puntaje_si, No = puntaje_no)


In [ ]:
posteriores <- c(Sí = puntaje_si, No = puntaje_no)
posteriores <- posteriores / sum(posteriores)
posteriores


Los productos iniciales son proporcionales a las probabilidades posteriores. Al dividir cada puntaje entre la suma de ambos se obtienen valores que suman uno y pueden interpretarse como probabilidades normalizadas bajo el modelo.

## El problema de las probabilidades iguales a cero

Cuando una combinación nunca aparece en el entrenamiento, una de las probabilidades condicionales puede ser cero. Al multiplicar, todo el puntaje de esa clase se vuelve cero.

Una solución habitual es el **suavizado de Laplace**, que agrega una pequeña cantidad a los conteos antes de calcular las probabilidades.

$$
\widehat{P}(X_j=x\mid C=c)
=\frac{n_{x,c}+\alpha}{n_c+\alpha k}
$$

Aquí, $k$ es el número de categorías y normalmente se utiliza $\alpha=1$.

## Naive Bayes gaussiano

Cuando un predictor es numérico continuo, una opción común es suponer que, dentro de cada clase, sigue una distribución normal:

$$
X_j\mid C=c\sim N(\mu_{jc},\sigma_{jc}^{2})
$$

La media y la desviación estándar se estiman por separado para cada clase. Esta variante se denomina **Naive Bayes gaussiano**.

## Instalar y cargar paquetes


In [ ]:
paquetes_nb <- c("readr", "dplyr", "ggplot2", "e1071")

faltantes_nb <- paquetes_nb[
  !vapply(paquetes_nb, requireNamespace, logical(1), quietly = TRUE)
]

if (length(faltantes_nb) > 0) {
  install.packages(faltantes_nb, repos = "https://cloud.r-project.org")
}

library(readr)
library(dplyr)
library(ggplot2)
library(e1071)
source("util_graficas.R")


## Cargar la base preparada de ATUS


In [ ]:
ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (!file.exists(ruta_atus_ml)) {
  stop(
    paste(
      "No se encontró datos/atus_ml_preparado.csv.",
      "Ejecute primero el capítulo de preparación de datos."
    )
  )
}

atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)


## Preparar una muestra para el modelo


In [ ]:
set.seed(123)

atus_nb <- atus_ml |>
  sample_n(min(20000, nrow(atus_ml))) |>
  transmute(
    accidente_con_victimas = factor(
      accidente_con_victimas,
      levels = c("Solo daños", "Con víctimas")
    ),
    MES = factor(MES),
    ID_HORA = as.numeric(ID_HORA),
    DIASEMANA = factor(DIASEMANA),
    TIPACCID = factor(TIPACCID),
    CAUSAACCI = factor(CAUSAACCI)
  ) |>
  na.omit()

prop.table(table(atus_nb$accidente_con_victimas))


## División estratificada en entrenamiento y prueba


In [ ]:
set.seed(123)

indices_nb <- unlist(
  lapply(
    split(seq_len(nrow(atus_nb)), atus_nb$accidente_con_victimas),
    function(indices) {
      sample(indices, size = floor(0.70 * length(indices)))
    }
  )
)

entrenamiento_nb <- atus_nb[indices_nb, ]
prueba_nb <- atus_nb[-indices_nb, ]

prop.table(table(entrenamiento_nb$accidente_con_victimas))
prop.table(table(prueba_nb$accidente_con_victimas))


## Entrenar el modelo Naive Bayes


In [ ]:
modelo_nb <- naiveBayes(
  accidente_con_victimas ~
    MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
  data = entrenamiento_nb,
  laplace = 1
)

modelo_nb


El argumento `laplace = 1` aplica suavizado de Laplace a los predictores categóricos.

## Obtener predicciones de clase


In [ ]:
prediccion_nb <- predict(
  modelo_nb,
  newdata = prueba_nb,
  type = "class"
)

head(prediccion_nb)


## Obtener probabilidades posteriores


In [ ]:
probabilidades_nb <- predict(
  modelo_nb,
  newdata = prueba_nb,
  type = "raw"
)

head(probabilidades_nb)


## Construir la matriz de confusión


In [ ]:
matriz_nb <- table(
  Real = prueba_nb$accidente_con_victimas,
  Predicho = prediccion_nb
)

matriz_nb


## Calcular las métricas


In [ ]:
division_segura_nb <- function(numerador, denominador) {
  if (is.na(denominador) || denominador == 0) {
    return(NA_real_)
  }

  numerador / denominador
}


In [ ]:
clase_positiva <- "Con víctimas"
clase_negativa <- "Solo daños"

vp <- matriz_nb[clase_positiva, clase_positiva]
fn <- matriz_nb[clase_positiva, clase_negativa]
fp <- matriz_nb[clase_negativa, clase_positiva]
vn <- matriz_nb[clase_negativa, clase_negativa]

exactitud <- division_segura_nb(vp + vn, vp + vn + fp + fn)
sensibilidad <- division_segura_nb(vp, vp + fn)
especificidad <- division_segura_nb(vn, vn + fp)
precision <- division_segura_nb(vp, vp + fp)
f1 <- if (is.na(precision) || is.na(sensibilidad) ||
          precision + sensibilidad == 0) {
  NA_real_
} else {
  2 * precision * sensibilidad / (precision + sensibilidad)
}

metricas_nb <- data.frame(
  Metrica = c(
    "Exactitud", "Sensibilidad", "Especificidad", "Precisión", "F1"
  ),
  Valor = c(exactitud, sensibilidad, especificidad, precision, f1)
)

metricas_nb


## Visualizar las métricas


In [ ]:
ggplot(metricas_nb, aes(x = reorder(Metrica, Valor), y = Valor)) +
  geom_col() +
  geom_text(
    aes(label = sprintf("%.3f", Valor)),
    hjust = -0.1,
    na.rm = TRUE
  ) +
  coord_flip() +
  scale_y_continuous(limits = c(0, 1.08)) +
  labs(
    title = "Desempeño del modelo Naive Bayes",
    x = NULL,
    y = "Valor"
  ) +
  tema_libro()


**Fuente:** elaboración propia mediante R con datos del INEGI, Estadística de Accidentes de Tránsito Terrestre en Zonas Urbanas y Suburbanas (ATUS), 2024.

## Inspeccionar las probabilidades aprendidas


In [ ]:
modelo_nb$apriori
modelo_nb$tables$MES


La primera salida muestra las probabilidades previas de las clases. La segunda resume las probabilidades condicionales estimadas para los meses.

## Ventajas de Naive Bayes

- Es sencillo y rápido de entrenar.
- Produce probabilidades de pertenencia a cada clase.
- Funciona con conjuntos de datos de alta dimensión.
- Puede combinar predictores categóricos y numéricos.
- Requiere menos datos que modelos más complejos.
- Constituye una excelente línea base probabilística.

## Limitaciones

- El supuesto de independencia condicional puede ser poco realista.
- Variables muy correlacionadas pueden contar información repetida.
- Las probabilidades estimadas no siempre están bien calibradas.
- Las categorías no observadas requieren suavizado.
- La forma gaussiana puede ser inadecuada para variables numéricas asimétricas.

Un buen valor de exactitud no demuestra que el supuesto de independencia sea verdadero. El modelo debe evaluarse con datos no utilizados en el entrenamiento y con métricas apropiadas para la clase de interés.

## Comparación conceptual con otros algoritmos

| Método | Idea principal | Escalamiento | Interpretabilidad | Relaciones no lineales |
|---|---|---:|---:|---:|
| Regresión logística | Modela probabilidades mediante una función logística | Recomendable | Alta | Limitada sin transformaciones |
| k-NN | Clasifica por cercanía | Necesario | Media | Sí |
| Árbol | Divide el espacio mediante reglas | No necesario | Alta | Sí |
| Random Forest | Combina muchos árboles | No necesario | Media | Sí |
| SVM | Maximiza el margen | Muy recomendable | Media-baja | Sí, mediante kernels |
| Naive Bayes | Multiplica probabilidades condicionales | Depende de la variante | Alta | Mediante distribuciones por clase |

## Actividades para el lector

1. Cambie el suavizado de Laplace a `0`, `0.5` y `2`. Compare las métricas.
2. Elimine una variable predictora y examine si mejora la sensibilidad.
3. Modifique el tamaño de la muestra y mida el tiempo de entrenamiento.
4. Compare Naive Bayes con la regresión logística utilizando exactamente la misma partición.
5. Examine qué categorías presentan probabilidades condicionales muy distintas entre las clases.
6. Explique por qué dos variables correlacionadas pueden influir doblemente en el producto de verosimilitudes.

## Conclusiones

Naive Bayes muestra que un modelo relativamente simple puede construir clasificaciones útiles mediante probabilidades. Su supuesto de independencia facilita el cálculo y reduce el costo computacional, aunque exige interpretar los resultados con prudencia.

En ATUS, el método proporciona una línea base probabilística rápida para predecir si un accidente pertenece a la clase con víctimas o a la clase de solo daños. Su desempeño debe juzgarse junto con la sensibilidad, la especificidad y F1, no únicamente por la exactitud.

## Referencias fundamentales de Naive Bayes

El análisis de las condiciones bajo las cuales el clasificador bayesiano simple puede ser óptimo fue desarrollado por @domingos1997optimality. Una discusión crítica sobre por qué el método puede funcionar bien pese a su supuesto ingenuo aparece en @hand2001idiots. Para una introducción moderna dentro del aprendizaje estadístico puede consultarse @james2021islr.

## Laboratorio interactivo: probabilidades posteriores Naive Bayes

Modifica probabilidades previas y condicionales para observar el resultado.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

Permite modificar probabilidades previas y condicionales.

## Caso aplicado B: Naive Bayes con COVID-19

Ahora aplicamos Naive Bayes a COVID-19 México 2022. La variable objetivo es `MURIO` y utilizamos la misma familia de predictores empleada en los capítulos anteriores.

> **Uso académico:** las probabilidades estimadas por este modelo no deben interpretarse como riesgo clínico individual.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_nb <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
                  OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na() |>
    dplyr::mutate(
      MURIO = factor(MURIO, levels = c(0, 1),
                     labels = c("Sin defunción", "Defunción")),
      NEUMONIA = factor(NEUMONIA),
      DIABETES = factor(DIABETES),
      HIPERTENSION = factor(HIPERTENSION),
      OBESIDAD = factor(OBESIDAD),
      RENAL_CRONICA = factor(RENAL_CRONICA)
    )

  set.seed(2026)
  covid_nb <- covid_nb |>
    dplyr::sample_n(min(12000, nrow(covid_nb)))

  set.seed(2026)
  idx_nb_covid <- unlist(lapply(
    split(seq_len(nrow(covid_nb)), covid_nb$MURIO),
    function(i) sample(i, floor(0.80 * length(i)))
  ))

  train_nb_covid <- covid_nb[idx_nb_covid, ]
  test_nb_covid <- covid_nb[-idx_nb_covid, ]
}


En este ejemplo las comorbilidades binarias se tratan como variables categóricas, mientras que `EDAD` y `NUM_COMORBILIDADES` se modelan como numéricas.


In [ ]:
if (exists("train_nb_covid")) {
  modelo_nb_covid <- e1071::naiveBayes(
    MURIO ~ ., data = train_nb_covid,
    laplace = 1
  )

  pred_nb_covid <- predict(modelo_nb_covid, test_nb_covid, type = "class")
  prob_nb_covid <- predict(modelo_nb_covid, test_nb_covid, type = "raw")

  matriz_nb_covid <- table(
    Real = test_nb_covid$MURIO,
    Predicho = pred_nb_covid
  )
  matriz_nb_covid
  head(prob_nb_covid)
}


In [ ]:
if (exists("matriz_nb_covid")) {
  VP <- matriz_nb_covid["Defunción", "Defunción"]
  FN <- matriz_nb_covid["Defunción", "Sin defunción"]
  FP <- matriz_nb_covid["Sin defunción", "Defunción"]
  VN <- matriz_nb_covid["Sin defunción", "Sin defunción"]
  div <- function(a,b) ifelse(b == 0, NA_real_, as.numeric(a/b))
  data.frame(
    exactitud = div(VP+VN, sum(matriz_nb_covid)),
    sensibilidad = div(VP, VP+FN),
    especificidad = div(VN, VN+FP),
    precision = div(VP, VP+FP)
  )
}


Naive Bayes permite observar directamente probabilidades previas y condicionales. Eso lo convierte en un excelente modelo didáctico para conectar el teorema de Bayes con un problema real de clasificación.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=UbfJ3bLDwUI) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-11/capitulo-11-naive-bayes-presentacion.pdf) | [Descargar PDF](recursos/capitulo-11/capitulo-11-naive-bayes-presentacion.pdf){download="capitulo-11-naive-bayes-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-11/capitulo-11-naive-bayes-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-11/capitulo-11-naive-bayes-presentacion.pptx){download="capitulo-11-naive-bayes-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-11/capitulo-11-naive-bayes-infografia.png) | [Descargar PNG](recursos/capitulo-11/capitulo-11-naive-bayes-infografia.png){download="capitulo-11-naive-bayes-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/11-naive-bayes.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 11](recursos/capitulo-11/capitulo-11-naive-bayes-infografia.png)](recursos/capitulo-11/capitulo-11-naive-bayes-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/11-naive-bayes.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=UbfJ3bLDwUI>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)


# Redes neuronales artificiales

La formulación matemática de **álgebra lineal, cálculo, descenso por gradiente y redes neuronales** se desarrolla con mayor profundidad
en los capítulos 2, 4 y 16 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos de aprendizaje

Al finalizar este capítulo podrás:

- explicar la estructura básica de una neurona artificial;
- interpretar entradas, pesos, sesgo y funciones de activación;
- describir el funcionamiento de una red neuronal multicapa;
- comprender la propagación hacia adelante y la retropropagación;
- distinguir entre clasificación binaria y multiclase;
- preparar correctamente los datos para una red neuronal;
- implementar una red sencilla desde cero en R;
- entrenar modelos con `neuralnet`;
- construir una red moderna con `keras`;
- evaluar el desempeño mediante matriz de confusión y métricas;
- reconocer sobreajuste, regularización y errores frecuentes.

## Introducción

Las **redes neuronales artificiales** son modelos computacionales inspirados, de forma muy simplificada, en la manera en que las neuronas biológicas reciben, transforman y transmiten señales.

Una red neuronal puede aprender relaciones complejas entre variables de entrada y una variable de respuesta. Esto la hace útil para:

- clasificación;
- regresión;
- reconocimiento de imágenes;
- procesamiento de señales;
- detección de patrones;
- predicción de series;
- aplicaciones biomédicas y ambientales.

La idea central es construir muchas unidades simples, llamadas **neuronas artificiales**, conectadas entre sí mediante pesos ajustables.

## La neurona artificial

Una neurona recibe varias entradas:

$$
x_1,x_2,\ldots,x_p
$$

Cada entrada tiene un peso:

$$
w_1,w_2,\ldots,w_p
$$

La neurona calcula una combinación lineal:

$$
z=w_1x_1+w_2x_2+\cdots+w_px_p+b
$$

donde $b$ es el **sesgo**.

Después aplica una función de activación:

$$
a=f(z)
$$

El resultado $a$ se transmite a otras neuronas o se utiliza como salida.

### Interpretación

- Las entradas son las características del problema.
- Los pesos indican la importancia de cada entrada.
- El sesgo permite desplazar la frontera de decisión.
- La función de activación introduce no linealidad.
- La salida representa una predicción o una señal intermedia.

## Ejemplo manual de una neurona

Supongamos que las entradas, los pesos y el sesgo son:

$$
\begin{aligned}
x_1 &= 0.8, & x_2 &= 0.4,\\
w_1 &= 1.2, & w_2 &= -0.7, & b &= 0.1.
\end{aligned}
$$

La combinación lineal se obtiene sustituyendo los valores:

$$
\begin{aligned}
z &= w_1x_1+w_2x_2+b\\
  &= (1.2)(0.8)+(-0.7)(0.4)+0.1\\
  &= 0.96-0.28+0.1\\
  &= 0.78.
\end{aligned}
$$

Si usamos la función sigmoide:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

obtenemos:

$$
\sigma(0.78)\approx0.686
$$

La neurona produce una salida cercana a 0.686.

## Funciones de activación

Las funciones de activación permiten que la red represente relaciones no lineales.

### Función escalón

$$
f(z)=
\begin{cases}
1, & z\geq0\\
0, & z<0
\end{cases}
$$

Fue utilizada en los primeros perceptrones.

### Función sigmoide

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Características:

- produce valores entre 0 y 1;
- es útil en clasificación binaria;
- puede interpretarse como probabilidad;
- puede saturarse para valores grandes.

### Tangente hiperbólica

$$
\tanh(z)=\frac{e^z-e^{-z}}{e^z+e^{-z}}
$$

Produce valores entre -1 y 1.

### ReLU

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

Es una de las activaciones más utilizadas en capas ocultas.

### Softmax

Para clasificación multiclase:

$$
P(y=k)=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}
$$

Convierte las salidas en probabilidades que suman 1.

## El perceptrón

El perceptrón es uno de los modelos neuronales más sencillos.

Calcula:

$$
\widehat{y}=f(\mathbf{w}^{T}\mathbf{x}+b)
$$

y actualiza los pesos cuando comete errores.

La regla básica de actualización es:

$$
w_j^{nuevo}=w_j^{anterior}+\eta(y-\widehat{y})x_j
$$

donde:

- $\eta$ es la tasa de aprendizaje;
- $y$ es la clase real;
- $\widehat{y}$ es la predicción.

### Limitación

Un perceptrón simple solo resuelve problemas linealmente separables. El problema XOR es el ejemplo clásico que requiere una capa oculta.

## Estructura de una red multicapa

Una red neuronal multicapa suele tener:

1. capa de entrada;
2. una o más capas ocultas;
3. capa de salida.

Cada neurona de una capa puede conectarse con las neuronas de la siguiente.

### Capa de entrada

Recibe las variables predictoras.

### Capas ocultas

Aprenden combinaciones intermedias de las características.

### Capa de salida

Produce la predicción final.

Ejemplos:

- una neurona sigmoide para clasificación binaria;
- varias neuronas softmax para clasificación multiclase;
- una neurona lineal para regresión.

## Propagación hacia adelante

La **propagación hacia adelante** calcula la salida de la red a partir de las entradas.

Para una capa oculta:

$$
\mathbf{h}=f(\mathbf{W}^{(1)}\mathbf{x}+\mathbf{b}^{(1)})
$$

Para la salida:

$$
\widehat{\mathbf{y}}=g(\mathbf{W}^{(2)}\mathbf{h}+\mathbf{b}^{(2)})
$$

La red transforma sucesivamente la información hasta producir una predicción.

## Función de pérdida

La función de pérdida mide qué tan lejos está la predicción del valor real.

### Error cuadrático medio

$$
MSE=\frac{1}{n}\sum_{i=1}^{n}(y_i-\widehat{y}_i)^2
$$

Se usa principalmente en regresión.

### Entropía cruzada binaria

$$
L=-\frac{1}{n}\sum_{i=1}^{n}
\left[
y_i\log(\widehat{p}_i)
+(1-y_i)\log(1-\widehat{p}_i)
\right]
$$

Se utiliza en clasificación binaria.

### Entropía cruzada categórica

$$
L=-\frac{1}{n}\sum_{i=1}^{n}\sum_{k=1}^{K}
y_{ik}\log(\widehat{p}_{ik})
$$

Se utiliza en clasificación multiclase.

## Descenso de gradiente

El entrenamiento busca valores de los pesos que minimicen la pérdida.

La actualización general es:

$$
w^{\mathrm{nuevo}}
=
w^{\mathrm{anterior}}
-
\eta\frac{\partial L}{\partial w}
$$

donde $\eta$ es la tasa de aprendizaje.

### Tasa demasiado pequeña

- aprendizaje lento;
- muchas iteraciones.

### Tasa demasiado grande

- oscilaciones;
- divergencia;
- pérdida inestable.

## Retropropagación

La **retropropagación** calcula cómo contribuye cada peso al error.

El proceso es:

1. realizar propagación hacia adelante;
2. calcular la pérdida;
3. obtener derivadas mediante la regla de la cadena;
4. propagar el error hacia atrás;
5. actualizar pesos y sesgos;
6. repetir.

La retropropagación no es un modelo distinto, sino el mecanismo principal para entrenar redes multicapa.

## Implementación manual de una neurona en R


In [ ]:
sigmoide <- function(z) {
  1 / (1 + exp(-z))
}

x <- c(0.8, 0.4)
w <- c(1.2, -0.7)
b <- 0.1

z <- sum(x * w) + b
salida <- sigmoide(z)

z
salida


## Laboratorio interactivo: construye una neurona artificial

El siguiente laboratorio se ejecuta directamente en tu navegador mediante **Shinylive**. No requiere un servidor Shiny externo.

Puedes modificar las entradas, los pesos, el sesgo, la función de activación y el umbral de clasificación.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


1. Mantén las entradas fijas y cambia solamente el sesgo.
2. Cambia el signo de uno de los pesos.
3. Compara sigmoide, ReLU y tangente hiperbólica.
4. Coloca ambos pesos en cero.
5. Modifica el umbral y observa si cambia la clase.

### Laboratorio interactivo disponible en la versión web

La versión web del capítulo incluye un laboratorio donde pueden modificarse las entradas, los pesos, el sesgo, la función de activación y el umbral de clasificación de una neurona artificial.

## Perceptrón desde cero en R


In [ ]:
entrenar_perceptron <- function(X, y,
                               eta = 0.1,
                               epocas = 100) {

  X <- as.matrix(X)
  y <- as.numeric(y)

  pesos <- rep(0, ncol(X))
  sesgo <- 0

  for (epoca in seq_len(epocas)) {

    for (i in seq_len(nrow(X))) {

      z <- sum(X[i, ] * pesos) + sesgo
      pred <- ifelse(z >= 0, 1, 0)
      error <- y[i] - pred

      pesos <- pesos + eta * error * X[i, ]
      sesgo <- sesgo + eta * error
    }
  }

  list(
    pesos = pesos,
    sesgo = sesgo
  )
}


Ejemplo con la compuerta AND:


In [ ]:
X_and <- data.frame(
  x1 = c(0, 0, 1, 1),
  x2 = c(0, 1, 0, 1)
)

y_and <- c(0, 0, 0, 1)

modelo_and <- entrenar_perceptron(
  X_and,
  y_and,
  eta = 0.1,
  epocas = 20
)

modelo_and


Función de predicción:


In [ ]:
predecir_perceptron <- function(modelo, X) {

  X <- as.matrix(X)

  z <- as.numeric(
    X %*% modelo$pesos +
      modelo$sesgo
  )

  ifelse(z >= 0, 1, 0)
}

predecir_perceptron(
  modelo_and,
  X_and
)


## Ejemplo de XOR

La compuerta XOR produce:

| $x_1$ | $x_2$ | XOR |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

No puede resolverse mediante una sola frontera lineal. Una capa oculta permite construir regiones más complejas.

## Preparación de los datos

Antes de entrenar una red neuronal es recomendable:

- tratar valores faltantes;
- convertir categorías;
- separar entrenamiento, validación y prueba;
- normalizar variables numéricas;
- revisar desbalance;
- eliminar identificadores;
- evitar fuga de información.

## Normalización

Las redes suelen entrenarse mejor cuando las variables tienen escalas comparables.

### Estandarización

$$
z=\frac{x-\bar{x}}{s}
$$

### Mínimo-máximo

$$
x^\ast=\frac{x-\min(x)}{\max(x)-\min(x)}
$$

Los parámetros deben calcularse solo con entrenamiento.


In [ ]:
set.seed(123)

indice <- sample(
  seq_len(nrow(iris)),
  size = round(0.70 * nrow(iris))
)

train <- iris[indice, ]
test <- iris[-indice, ]

medias <- sapply(train[, 1:4], mean)
desvios <- sapply(train[, 1:4], sd)

x_train <- scale(
  train[, 1:4],
  center = medias,
  scale = desvios
)

x_test <- scale(
  test[, 1:4],
  center = medias,
  scale = desvios
)


## Clasificación binaria con neuralnet

Para que la generación del libro sea estable, los bloques que entrenan la red con `neuralnet` se muestran, pero no se ejecutan automáticamente durante el renderizado. Puedes ejecutarlos en RStudio después de instalar el paquete con `PREPARAR_PAQUETES_RED_NEURONAL.bat`.

Crearemos un problema con dos especies de `iris`.


In [ ]:
iris_bin <- subset(
  iris,
  Species != "setosa"
)

iris_bin$clase <- ifelse(
  iris_bin$Species == "virginica",
  1,
  0
)

iris_bin$Species <- NULL


División:


In [ ]:
set.seed(321)

idx <- sample(
  seq_len(nrow(iris_bin)),
  size = round(0.70 * nrow(iris_bin))
)

train_bin <- iris_bin[idx, ]
test_bin <- iris_bin[-idx, ]


Normalización segura:


In [ ]:
variables <- setdiff(
  names(train_bin),
  "clase"
)

medias_bin <- sapply(
  train_bin[, variables],
  mean
)

desvios_bin <- sapply(
  train_bin[, variables],
  sd
)

train_bin[, variables] <- scale(
  train_bin[, variables],
  center = medias_bin,
  scale = desvios_bin
)

test_bin[, variables] <- scale(
  test_bin[, variables],
  center = medias_bin,
  scale = desvios_bin
)


Entrenamiento:


## Predicción con neuralnet


## Métricas binarias


## Selección del umbral

El umbral 0.5 no siempre es el mejor.


## Visualizador interactivo de una red neuronal

Este laboratorio representa gráficamente una arquitectura similar a la que se especifica con el argumento `hidden` de `neuralnet`. Modifica el número de neuronas de las capas ocultas y observa cómo cambian la topología y el número de conexiones.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


El visualizador no entrena el modelo dentro del navegador. Su propósito es mostrar la arquitectura definida mediante `hidden`. Cuando ejecutes `plot(modelo_rna)` en RStudio, `neuralnet` dibujará además los pesos y sesgos aprendidos.

### Visualizador disponible en la versión web

La versión web permite modificar el número de entradas, neuronas ocultas y salidas, y genera dinámicamente el dibujo de la red y el código equivalente para `neuralnet`.

## Arquitectura de la red

Una arquitectura se define por:

- número de capas ocultas;
- neuronas por capa;
- activaciones;
- conexiones;
- regularización.

No existe una arquitectura universal.

### Red muy pequeña

Puede producir subajuste.

### Red demasiado grande

Puede memorizar entrenamiento y sobreajustar.

Una estrategia razonable es comenzar con una red pequeña y aumentar complejidad gradualmente.

## Épocas, lotes y optimizadores

### Época

Una época representa una pasada completa por entrenamiento.

### Lote

Un lote es un subconjunto utilizado para calcular una actualización.

### Descenso por lotes

Usa todo el conjunto.

### Descenso estocástico

Actualiza con una observación.

### Mini-batch

Usa pequeños grupos y es la opción más común.

### Optimizadores

Entre los más conocidos se encuentran:

- SGD;
- Momentum;
- RMSprop;
- Adam.

## Sobreajuste

Una red sobreajustada obtiene:

- error bajo en entrenamiento;
- error alto en validación o prueba.

Señales:

- la pérdida de entrenamiento sigue bajando;
- la pérdida de validación comienza a subir;
- crece la diferencia entre ambas.

## Regularización

### Regularización L2

Agrega una penalización:

$$
L_{total}=L+\lambda\sum_j w_j^2
$$

### Regularización L1

$$
L_{total}=L+\lambda\sum_j |w_j|
$$

### Dropout

Desactiva aleatoriamente una proporción de neuronas durante el entrenamiento.

### Early stopping

Detiene el entrenamiento cuando la validación deja de mejorar.

## Implementación moderna con keras

El paquete `keras` permite construir redes utilizando una interfaz de alto nivel.

La instalación de `keras` y su motor puede variar según el equipo. Conviene ejecutar este ejemplo en un entorno configurado o en Google Colab.


Modelo:


Compilación:


Entrenamiento:


Evaluación:


## Curvas de aprendizaje

Las curvas de pérdida y exactitud ayudan a diagnosticar el entrenamiento.


Debe compararse:

- desempeño de entrenamiento;
- desempeño de validación;
- punto donde comienza el sobreajuste.

## Clasificación multiclase

En un problema con $K$ clases:

- la capa de salida suele tener $K$ neuronas;
- se utiliza softmax;
- la clase predicha es la de mayor probabilidad.

$$
\widehat{y}=
\arg\max_k P(y=k\mid\mathbf{x})
$$

## Codificación de la respuesta

Para clasificación multiclase pueden emplearse:

- enteros de 0 a $K-1$;
- variables indicadoras one-hot.

Ejemplo one-hot:

| Clase | Neurona 1 | Neurona 2 | Neurona 3 |
|---|---:|---:|---:|
| A | 1 | 0 | 0 |
| B | 0 | 1 | 0 |
| C | 0 | 0 | 1 |

## Inicialización de pesos

Todos los pesos no deben comenzar con el mismo valor, porque las neuronas aprenderían exactamente lo mismo.

Las inicializaciones modernas buscan:

- romper simetría;
- mantener estable la varianza;
- mejorar el flujo de gradientes.

## Gradientes que desaparecen o explotan

En redes profundas, los gradientes pueden:

- aproximarse a cero;
- crecer excesivamente.

Consecuencias:

- aprendizaje muy lento;
- inestabilidad;
- pesos extremos.

Soluciones comunes:

- ReLU;
- inicialización adecuada;
- normalización;
- arquitecturas apropiadas;
- control de la tasa de aprendizaje.

## Importancia de la reproducibilidad

Para reproducir resultados:

- fijar semillas;
- registrar arquitectura;
- guardar parámetros;
- documentar divisiones;
- conservar versiones;
- guardar modelos entrenados;
- anotar métricas y umbrales.


In [ ]:
set.seed(2026)


## Interpretabilidad

Las redes neuronales suelen considerarse menos interpretables que:

- regresión logística;
- árboles de decisión;
- modelos lineales.

Herramientas posibles:

- importancia por permutación;
- análisis de sensibilidad;
- gráficos de dependencia;
- SHAP;
- LIME;
- inspección de activaciones.

La interpretación debe acompañarse de validación y conocimiento del dominio.

## Aplicación biomédica

Ejemplo: clasificar pacientes según riesgo de enfermedad.

Variables:

- edad;
- presión arterial;
- glucosa;
- colesterol;
- frecuencia cardiaca.

Respuesta:

- riesgo bajo;
- riesgo alto.

Debe prestarse especial atención a:

- sensibilidad;
- falsos negativos;
- calidad de los datos;
- validación externa;
- sesgo de selección;
- explicabilidad.

## Aplicación ambiental

Ejemplo: clasificar niveles de contaminación.

Variables:

- PM10;
- ozono;
- temperatura;
- humedad;
- velocidad del viento;
- hora;
- estación.

Respuesta:

- aceptable;
- no aceptable.

La red puede capturar relaciones no lineales, pero debe compararse con modelos más simples.

## Comparación con otros algoritmos

| Aspecto | Red neuronal | Regresión logística | Árbol | Random Forest |
|---|---|---|---|---|
| No linealidad | Alta | Limitada | Alta | Alta |
| Interpretación | Baja | Alta | Alta | Media |
| Preparación | Alta | Media | Media | Media |
| Escalamiento | Importante | Recomendable | Poco importante | Poco importante |
| Datos requeridos | Frecuentemente muchos | Menos | Moderados | Moderados |
| Costo | Puede ser alto | Bajo | Bajo | Moderado |

## Ventajas

- modelan relaciones complejas;
- representan no linealidades;
- se adaptan a clasificación y regresión;
- pueden aprender características internas;
- escalan a problemas de gran dimensión;
- son base de aprendizaje profundo.

## Limitaciones

- requieren preparación cuidadosa;
- pueden necesitar muchos datos;
- implican varios hiperparámetros;
- pueden sobreajustar;
- son menos interpretables;
- el entrenamiento puede ser costoso;
- los resultados dependen de arquitectura e inicialización.

## Errores frecuentes

1. No normalizar variables.
2. Usar el conjunto de prueba para ajustar.
3. Entrenar demasiadas épocas sin validación.
4. Usar una red enorme para pocos datos.
5. Evaluar solo exactitud.
6. Ignorar el desbalance.
7. No fijar semilla.
8. No guardar la arquitectura.
9. Interpretar probabilidades como certezas.
10. Comparar modelos con particiones diferentes.
11. Dejar identificadores como predictores.
12. No comparar contra un modelo simple.

## Flujo de trabajo recomendado

1. Definir el objetivo.
2. Preparar los datos.
3. Separar entrenamiento, validación y prueba.
4. Normalizar con entrenamiento.
5. Construir una arquitectura pequeña.
6. Entrenar y revisar curvas.
7. Ajustar hiperparámetros.
8. Aplicar regularización.
9. Evaluar en prueba una sola vez.
10. Comparar con modelos base.
11. Interpretar errores.
12. Documentar y guardar el modelo.

## Actividad guiada

Utiliza `iris` para clasificación multiclase.

1. Divide los datos en entrenamiento y prueba.
2. Normaliza las cuatro variables.
3. Diseña una red con una capa oculta.
4. Entrena el modelo.
5. Obtén probabilidades.
6. Asigna la clase más probable.
7. Construye la matriz de confusión.
8. Calcula exactitud.
9. Repite con más neuronas.
10. Compara los resultados.
11. Describe si existe sobreajuste.
12. Compara contra k-NN o Random Forest.

## Ejercicios

### Ejercicio 1

Calcula manualmente la salida de una neurona sigmoide.

### Ejercicio 2

Programa una neurona ReLU en R.

### Ejercicio 3

Entrena un perceptrón para la compuerta OR.

### Ejercicio 4

Explica por qué XOR requiere una capa oculta.

### Ejercicio 5

Entrena una red binaria con `neuralnet`.

### Ejercicio 6

Compara arquitecturas `hidden = 3`, `hidden = 5` y `hidden = c(5,3)`.

### Ejercicio 7

Modifica el umbral de clasificación y analiza sensibilidad y especificidad.

### Ejercicio 8

Construye una red multiclase con `keras`.

### Ejercicio 9

Compara la red con regresión logística.

### Ejercicio 10

Describe tres medidas para reducir sobreajuste.

## Preguntas de reflexión

- ¿Qué función cumple el sesgo?
- ¿Por qué se necesitan activaciones no lineales?
- ¿Qué diferencia existe entre propagación hacia adelante y retropropagación?
- ¿Por qué debe separarse validación de prueba?
- ¿Qué indica una pérdida de validación creciente?
- ¿Cuándo conviene usar una red neuronal?
- ¿Cuándo sería preferible un modelo más simple?
- ¿Por qué las probabilidades deben interpretarse con cautela?

## Caso aplicado B: red neuronal con COVID-19

En este ejemplo entrenamos una red neuronal pequeña para clasificar `MURIO` con datos de COVID-19 México 2022. Se utiliza una arquitectura deliberadamente sencilla para que el objetivo sea comprender el procedimiento y no construir un sistema clínico.

> **Uso académico:** la salida de la red es una predicción estadística sobre esta muestra y no un pronóstico médico individual.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_nn <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
                  OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_nn <- covid_nn |>
    dplyr::sample_n(min(8000, nrow(covid_nn)))

  set.seed(2026)
  idx_nn_covid <- unlist(lapply(
    split(seq_len(nrow(covid_nn)), covid_nn$MURIO),
    function(i) sample(i, floor(0.80 * length(i)))
  ))

  train_nn_covid <- covid_nn[idx_nn_covid, ]
  test_nn_covid <- covid_nn[-idx_nn_covid, ]

  predictores_nn <- c("EDAD", "NEUMONIA", "DIABETES", "HIPERTENSION",
                      "OBESIDAD", "RENAL_CRONICA", "NUM_COMORBILIDADES")

  medias_nn <- sapply(train_nn_covid[predictores_nn], mean)
  desv_nn <- sapply(train_nn_covid[predictores_nn], sd)
  desv_nn[desv_nn == 0] <- 1

  train_nn_covid[predictores_nn] <- scale(
    train_nn_covid[predictores_nn], center = medias_nn, scale = desv_nn
  )
  test_nn_covid[predictores_nn] <- scale(
    test_nn_covid[predictores_nn], center = medias_nn, scale = desv_nn
  )
}


La estandarización se calcula exclusivamente con el conjunto de entrenamiento y después se aplica al conjunto de prueba. Esto evita utilizar información de prueba durante el aprendizaje.


In [ ]:
if (exists("train_nn_covid")) {
  set.seed(2026)
  modelo_nn_covid <- neuralnet::neuralnet(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_nn_covid,
    hidden = c(5, 3),
    linear.output = FALSE,
    lifesign = "none",
    stepmax = 1e6
  )

  prob_nn_covid <- as.numeric(
    neuralnet::compute(
      modelo_nn_covid,
      test_nn_covid[predictores_nn]
    )$net.result[, 1]
  )

  pred_nn_covid <- ifelse(prob_nn_covid >= 0.50, 1, 0)
  matriz_nn_covid <- table(
    Real = factor(test_nn_covid$MURIO, levels = c(0,1), labels = c("Sin defunción", "Defunción")),
    Predicho = factor(pred_nn_covid, levels = c(0,1), labels = c("Sin defunción", "Defunción"))
  )
  matriz_nn_covid
}


In [ ]:
if (exists("matriz_nn_covid")) {
  VP <- matriz_nn_covid["Defunción", "Defunción"]
  FN <- matriz_nn_covid["Defunción", "Sin defunción"]
  FP <- matriz_nn_covid["Sin defunción", "Defunción"]
  VN <- matriz_nn_covid["Sin defunción", "Sin defunción"]
  div <- function(a,b) ifelse(b == 0, NA_real_, as.numeric(a/b))
  data.frame(
    exactitud = div(VP+VN, sum(matriz_nn_covid)),
    sensibilidad = div(VP, VP+FN),
    especificidad = div(VN, VN+FP)
  )
}


La red puede representar relaciones no lineales entre edad y comorbilidades, pero eso no garantiza que supere a modelos más simples. La comparación con regresión logística, k-NN, árboles, Random Forest, SVM y Naive Bayes será más importante que observar una sola métrica.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=gk3bt0h3WUU) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pdf) | [Descargar PDF](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pdf){download="capitulo-12-redes-neuronales-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-12/capitulo-12-redes-neuronales-presentacion.pptx){download="capitulo-12-redes-neuronales-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png) | [Descargar PNG](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png){download="capitulo-12-redes-neuronales-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/12-redes-neuronales.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 12](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png)](recursos/capitulo-12/capitulo-12-redes-neuronales-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/12-redes-neuronales.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=gk3bt0h3WUU>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

Las redes neuronales artificiales combinan neuronas simples para aprender relaciones complejas.

Sus componentes fundamentales son:

- entradas;
- pesos;
- sesgos;
- funciones de activación;
- capas;
- función de pérdida;
- optimización;
- retropropagación.

Una red efectiva no depende solamente de aumentar capas o neuronas. También requiere:

- datos de calidad;
- normalización correcta;
- validación;
- regularización;
- selección cuidadosa de arquitectura;
- evaluación con métricas apropiadas;
- comparación con modelos más sencillos.

En el siguiente capítulo podremos abordar métodos no supervisados, comenzando con **agrupamiento k-means**.


# Agrupamiento con k-means

La formulación matemática de **distancias, función objetivo, centroides y agrupamiento k-means** se desarrolla con mayor profundidad
en los capítulos 2 y 18 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos de aprendizaje

Al finalizar este capítulo podrás:

- distinguir aprendizaje supervisado y no supervisado;
- explicar intuitivamente el algoritmo k-means;
- calcular centroides y asignar observaciones a grupos;
- implementar k-means manualmente y con `kmeans()`;
- seleccionar el número de grupos mediante el método del codo y silhouette;
- interpretar centroides y perfiles de los grupos;
- reconocer las limitaciones y los errores frecuentes;
- aplicar el algoritmo a datos reales.

## Del aprendizaje supervisado al no supervisado

En los capítulos anteriores conocíamos la variable de respuesta. En el aprendizaje no supervisado no existe una etiqueta conocida que indique el grupo correcto. El objetivo es descubrir estructura en los datos.

El **agrupamiento** o *clustering* reúne observaciones similares y separa observaciones diferentes.

## Idea intuitiva de k-means

k-means busca dividir las observaciones en $k$ grupos. Cada grupo se representa mediante un **centroide**, que es el promedio de sus observaciones.

El algoritmo repite dos operaciones:

1. asignar cada observación al centroide más cercano;
2. recalcular los centroides.

El proceso termina cuando las asignaciones dejan de cambiar o la mejora es muy pequeña.

## Función objetivo

k-means minimiza la suma de cuadrados dentro de los grupos:

$$
WSS=\sum_{j=1}^{k}\sum_{\mathbf{x}_i\in C_j}\|\mathbf{x}_i-\boldsymbol{\mu}_j\|^2
$$

Aquí, $C_j$ es el grupo $j$ y $\boldsymbol{\mu}_j$ es su centroide.

## Ejemplo manual

Considera seis puntos:

| Punto | x | y |
|---|---:|---:|
| A | 1 | 1 |
| B | 1.5 | 2 |
| C | 3 | 4 |
| D | 5 | 7 |
| E | 3.5 | 5 |
| F | 4.5 | 5 |

Si fijamos $k=2$, elegimos dos centroides iniciales, asignamos cada punto al más cercano y recalculamos los promedios de ambos grupos. Repetimos hasta estabilizar.

## Implementación básica en R


In [ ]:
datos <- data.frame(
  x = c(1, 1.5, 3, 5, 3.5, 4.5),
  y = c(1, 2, 4, 7, 5, 5)
)

datos


In [ ]:
set.seed(123)
modelo_km <- kmeans(datos, centers = 2, nstart = 25)
modelo_km


In [ ]:
datos$cluster <- factor(modelo_km$cluster)
plot(
  datos$x, datos$y,
  col = datos$cluster,
  pch = 19,
  xlab = "x", ylab = "y",
  main = "Agrupamiento k-means"
)
points(modelo_km$centers, pch = 8, cex = 2, lwd = 2)


## ¿Por qué usar `nstart`?

k-means depende de los centroides iniciales. `nstart = 25` ejecuta el algoritmo varias veces y conserva la solución con menor variación interna.

## Escalamiento de variables

Si una variable está medida en miles y otra entre 0 y 10, la primera dominará las distancias. Por ello suele ser necesario estandarizar:

$$
z=\frac{x-\bar{x}}{s}
$$


In [ ]:
X <- scale(iris[, 1:4])
head(X)


## Aplicación con `iris`


In [ ]:
set.seed(2026)
km_iris <- kmeans(X, centers = 3, nstart = 50)

table(
  Cluster = km_iris$cluster,
  Especie = iris$Species
)


Los números de los clusters no tienen significado previo. El grupo 1 no es mejor ni peor que el grupo 2.

## Interpretación de centroides


In [ ]:
km_iris$centers


Cada fila representa un perfil promedio estandarizado. Valores positivos indican niveles superiores a la media; valores negativos, inferiores.

## Método del codo

El método del codo compara la suma de cuadrados interna para distintos valores de $k$.


In [ ]:
valores_k <- 1:10
wss <- numeric(length(valores_k))

for (i in seq_along(valores_k)) {
  set.seed(123)
  ajuste <- kmeans(X, centers = valores_k[i], nstart = 25)
  wss[i] <- ajuste$tot.withinss
}

plot(
  valores_k, wss,
  type = "b",
  xlab = "Número de clusters k",
  ylab = "Suma de cuadrados interna",
  main = "Método del codo"
)


Se busca un punto a partir del cual añadir grupos produce mejoras pequeñas.

## Coeficiente silhouette

Silhouette compara cohesión interna y separación externa. Sus valores se encuentran entre -1 y 1:

- cerca de 1: agrupamiento claro;
- cerca de 0: observación fronteriza;
- negativo: posible asignación incorrecta.


## Laboratorio interactivo k-means

Modifica el número de grupos y observa cómo cambian los centroides y la separación.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


El laboratorio interactivo de k-means está disponible en la versión web del capítulo. Permite modificar $k$, el tamaño de la muestra y la dispersión de los grupos.

## Ventajas

- sencillo e intuitivo;
- rápido en conjuntos medianos y grandes;
- fácil de visualizar;
- útil para segmentación y exploración;
- disponible en R base.

## Limitaciones

- requiere fijar $k$;
- depende de la inicialización;
- es sensible a escalas y valores atípicos;
- favorece grupos aproximadamente esféricos;
- no maneja bien clusters con densidades muy diferentes;
- solo trabaja naturalmente con variables numéricas.

## Errores frecuentes

1. No estandarizar variables.
2. Elegir $k$ únicamente porque coincide con una expectativa previa.
3. Ejecutar una sola inicialización.
4. Interpretar los números de cluster como categorías ordenadas.
5. Incluir identificadores.
6. Ignorar valores atípicos.
7. Confundir clusters con clases verdaderas.
8. Describir grupos sin revisar sus centroides.

## Aplicación con datos reales

En accidentes de tránsito pueden agruparse municipios, horarios o tipos de accidente usando variables como número de víctimas, frecuencia, hora, zona y severidad. El objetivo no es predecir una etiqueta, sino descubrir perfiles semejantes.

## Actividad guiada

1. Selecciona cuatro variables numéricas de un conjunto real.
2. Trata valores faltantes.
3. Estandariza los datos.
4. Evalúa valores de $k$ entre 2 y 10.
5. Usa el método del codo.
6. Calcula silhouette.
7. Entrena el modelo final con `nstart = 50`.
8. Interpreta los centroides.
9. Asigna nombres descriptivos a los perfiles.
10. Explica las limitaciones del análisis.

## Ejercicios

1. Implementa una iteración manual de k-means.
2. Compara resultados con y sin estandarización.
3. Cambia la semilla y analiza la estabilidad.
4. Agrega valores atípicos.
5. Compara $k=2$, $k=3$ y $k=5$.
6. Contrasta el método del codo y silhouette.
7. Aplica k-means a datos ambientales o de accidentes.

## Caso aplicado B: k-means con COVID-19

En esta segunda ruta aplicada usamos datos abiertos de COVID-19 México 2022 para descubrir **perfiles de observaciones semejantes**. A diferencia de los capítulos de clasificación, k-means es un método no supervisado: por ello la variable `MURIO` **no participa en la formación de los grupos**.

> **Uso académico:** los clusters representan patrones estadísticos dentro de esta muestra. No son categorías clínicas, diagnósticos ni niveles de riesgo médico.

### Preparar las variables para agrupamiento

Usamos variables numéricas relacionadas con edad y comorbilidades. Las variables binarias se mantienen como 0/1 y todas se estandarizan para evitar que una escala domine las distancias.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_km <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(
      MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
      OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES
    ) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_km <- covid_km |>
    dplyr::sample_n(min(12000, nrow(covid_km)))

  variables_km <- c(
    "EDAD", "NEUMONIA", "DIABETES", "HIPERTENSION",
    "OBESIDAD", "RENAL_CRONICA", "NUM_COMORBILIDADES"
  )

  X_covid_km <- scale(covid_km[variables_km])
}


`MURIO` se conserva únicamente para una interpretación posterior. No se incluye en `X_covid_km`, de modo que los clusters se forman sin conocer la variable de defunción.

### Explorar el número de grupos con el método del codo


In [ ]:
if (exists("X_covid_km")) {
  valores_k_covid <- 1:8
  wss_covid <- numeric(length(valores_k_covid))

  for (i in seq_along(valores_k_covid)) {
    set.seed(2026)
    ajuste <- kmeans(
      X_covid_km,
      centers = valores_k_covid[i],
      nstart = 30,
      iter.max = 100
    )
    wss_covid[i] <- ajuste$tot.withinss
  }

  plot(
    valores_k_covid, wss_covid,
    type = "b", pch = 19,
    xlab = "Número de clusters k",
    ylab = "Suma de cuadrados interna",
    main = "COVID-19: método del codo"
  )
}


El método del codo no elige automáticamente un valor perfecto de \(k\); ayuda a identificar cuándo agregar más clusters produce mejoras cada vez menores.

### Comparar silhouette

Para que el cálculo de distancias sea ligero, se evalúa silhouette sobre una submuestra reproducible.


In [ ]:
if (exists("X_covid_km")) {
  set.seed(2026)
  n_sil <- min(2500, nrow(X_covid_km))
  idx_sil <- sample(seq_len(nrow(X_covid_km)), n_sil)
  X_sil <- X_covid_km[idx_sil, , drop = FALSE]
  dist_sil <- dist(X_sil)

  sil_promedio <- sapply(2:6, function(k_actual) {
    set.seed(2026)
    km_tmp <- kmeans(X_sil, centers = k_actual, nstart = 30)
    sil_tmp <- cluster::silhouette(km_tmp$cluster, dist_sil)
    mean(sil_tmp[, "sil_width"])
  })

  data.frame(
    k = 2:6,
    silhouette_promedio = sil_promedio
  )
}


### Ajustar una solución didáctica con tres clusters

Para mantener la interpretación sencilla usamos \(k=3\). En un análisis formal, este valor debería justificarse conjuntamente con el codo, silhouette, estabilidad e interpretabilidad.


In [ ]:
if (exists("X_covid_km")) {
  set.seed(2026)
  modelo_covid_km <- kmeans(
    X_covid_km,
    centers = 3,
    nstart = 50,
    iter.max = 100
  )

  covid_km$cluster <- factor(modelo_covid_km$cluster)
  table(covid_km$cluster)
}


### Interpretar los centroides


In [ ]:
if (exists("modelo_covid_km")) {
  centroides_covid <- as.data.frame(modelo_covid_km$centers)
  centroides_covid$cluster <- factor(seq_len(nrow(centroides_covid)))
  centroides_covid
}


Como los datos están estandarizados, un centroide positivo indica que el cluster presenta, en promedio, valores superiores a la media de la muestra para esa variable; un valor negativo indica valores inferiores.

### Perfil descriptivo en unidades originales


In [ ]:
if (exists("modelo_covid_km")) {
  perfiles_covid <- covid_km |>
    dplyr::group_by(cluster) |>
    dplyr::summarise(
      n = dplyr::n(),
      edad_media = mean(EDAD),
      neumonia_pct = mean(NEUMONIA) * 100,
      diabetes_pct = mean(DIABETES) * 100,
      hipertension_pct = mean(HIPERTENSION) * 100,
      obesidad_pct = mean(OBESIDAD) * 100,
      renal_cronica_pct = mean(RENAL_CRONICA) * 100,
      comorbilidades_media = mean(NUM_COMORBILIDADES),
      .groups = "drop"
    )

  perfiles_covid
}


### Observar `MURIO` después de formar los grupos

Ahora sí usamos la variable `MURIO`, pero únicamente como una **descripción externa** de los clusters ya construidos.


In [ ]:
if (exists("modelo_covid_km")) {
  mortalidad_por_cluster <- covid_km |>
    dplyr::group_by(cluster) |>
    dplyr::summarise(
      casos = dplyr::n(),
      defunciones_registradas = sum(MURIO == 1),
      porcentaje_defuncion = mean(MURIO == 1) * 100,
      .groups = "drop"
    )

  mortalidad_por_cluster
}


Una diferencia en el porcentaje de defunción entre clusters **no demuestra causalidad** y tampoco convierte k-means en un modelo predictivo. Los grupos fueron definidos únicamente por semejanza en las variables de entrada.

### Visualización bidimensional mediante PCA

La siguiente gráfica usa PCA solo para proyectar los clusters en dos dimensiones y facilitar su visualización; el agrupamiento continúa siendo el realizado en el espacio estandarizado original.


In [ ]:
if (exists("modelo_covid_km")) {
  pca_covid_km <- prcomp(X_covid_km, center = FALSE, scale. = FALSE)

  grafica_covid_km <- data.frame(
    PC1 = pca_covid_km$x[, 1],
    PC2 = pca_covid_km$x[, 2],
    cluster = covid_km$cluster
  )

  ggplot2::ggplot(
    grafica_covid_km,
    ggplot2::aes(x = PC1, y = PC2, color = cluster)
  ) +
    ggplot2::geom_point(alpha = 0.35, size = 1.2) +
    ggplot2::labs(
      title = "Perfiles COVID-19 obtenidos con k-means",
      subtitle = "Proyección PCA para visualizar los clusters",
      x = "Componente principal 1",
      y = "Componente principal 2",
      color = "Cluster"
    ) +
    ggplot2::theme_minimal()
}


Este ejercicio muestra una diferencia fundamental: en clasificación usamos una etiqueta para aprender a predecirla; en clustering buscamos estructura sin etiqueta. Después podemos relacionar los grupos con variables externas para describirlos, pero no para afirmar que esas variables causaron los clusters.

## Laboratorio interactivo COVID: agrupamiento k-means

Este laboratorio utiliza una **submuestra reproducible de la muestra COVID-19 México 2022 incluida en el libro**. Puedes modificar el número de clusters y observar cómo cambian los perfiles en una proyección de componentes principales.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio interactivo COVID disponible en la versión web

La versión web permite cambiar el número de clusters y visualizar una submuestra COVID-19 2022 agrupada mediante k-means y proyectada con PCA.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=mZgUlnSxrgA) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-13/capitulo-13-kmeans-presentacion.pdf) | [Descargar PDF](recursos/capitulo-13/capitulo-13-kmeans-presentacion.pdf){download="capitulo-13-kmeans-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-13/capitulo-13-kmeans-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-13/capitulo-13-kmeans-presentacion.pptx){download="capitulo-13-kmeans-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-13/capitulo-13-kmeans-infografia.png) | [Descargar PNG](recursos/capitulo-13/capitulo-13-kmeans-infografia.png){download="capitulo-13-kmeans-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/13-kmeans.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 13](recursos/capitulo-13/capitulo-13-kmeans-infografia.png)](recursos/capitulo-13/capitulo-13-kmeans-infografia.png)

<!-- colab-capitulo -->

Este capítulo cuenta con un **cuaderno autónomo de Google Colab**. Puede abrirse y ejecutarse de manera independiente, sin necesidad de ejecutar los capítulos anteriores.

[**Abrir este capítulo en Google Colab**](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/13-kmeans.ipynb)

::: <!-- /colab-capitulo -->

**Video del capítulo:** <https://www.youtube.com/watch?v=mZgUlnSxrgA>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Conclusión

k-means descubre grupos mediante distancias y centroides. Su utilidad depende de una preparación adecuada, una selección razonada de $k$ y una interpretación basada en los perfiles de cada cluster.

Con este capítulo cerramos la ruta principal de algoritmos del libro. Los apéndices reúnen el glosario, las referencias y los índices para consulta.
